In [ ]:
import torch
print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = "/content/drive/MyDrive/oral_cancer_crosssite"
for sub in ["data/raw", "data/sample", "outputs/tables",
            "outputs/figures", "outputs/models", "src"]:
    os.makedirs(f"{ROOT}/{sub}", exist_ok=True)
print("ready:", ROOT)

In [ ]:
!pip install -q timm albumentations

In [ ]:
import os, glob, zipfile, pathlib, collections, shutil

SRC = f"{ROOT}/data/sample"
WORK = "/content/sample"

zips = glob.glob(f"{SRC}/*.zip")
print("found zips:", [os.path.basename(z) for z in zips])

if zips:
    shutil.rmtree(WORK, ignore_errors=True)
    os.makedirs(WORK, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(WORK)
    print("extracted to", WORK)
    SCAN = WORK
else:
    SCAN = SRC
    print("no zip found, scanning folder directly")

img_ext = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
ext_counts, per_dir, examples = collections.Counter(), collections.Counter(), {}
label_files = []

for dirpath, _, filenames in os.walk(SCAN):
    rel = os.path.relpath(dirpath, SCAN)
    for f in filenames:
        e = pathlib.Path(f).suffix.lower()
        ext_counts[e] += 1
        if e in img_ext:
            per_dir[rel] += 1
            examples.setdefault(rel, f)
        if e in {".csv", ".tsv", ".json", ".xml", ".txt", ".xlsx"}:
            label_files.append(os.path.join(rel, f))

print("\n=== file types ===")
for e, n in ext_counts.most_common(15):
    print(f"  {e or '(none)':10s} {n}")

print("\n=== folders with images ===")
for d, n in per_dir.most_common(40):
    print(f"  {n:6d}  {d}")
    print(f"          e.g. {examples[d]}")

print(f"\n=== label files ({len(label_files)}) ===")
for p in label_files[:25]:
    print("  ", p)

In [ ]:
import pandas as pd

meta = pd.read_excel(f"{SCAN}/metadata.xlsx")

print("shape:", meta.shape)
print("\n=== columns ===")
for c in meta.columns:
    nun = meta[c].nunique(dropna=True)
    nnull = meta[c].isna().sum()
    print(f"  {c!r:40s} unique={nun:<5} nulls={nnull}")

print("\n=== first 5 rows ===")
pd.set_option("display.max_columns", None, "display.width", 250)
print(meta.head())

print("\n=== value counts for low-cardinality columns ===")
for c in meta.columns:
    if meta[c].nunique(dropna=True) <= 15:
        print(f"\n-- {c} --")
        print(meta[c].value_counts(dropna=False))

In [ ]:
import pandas as pd, re, os

meta = pd.read_excel(f"{SCAN}/metadata.xlsx")

# actual files on disk, keyed by lowercase name so .jpg/.jpeg mismatches don't bite
on_disk = {f.lower(): f for f in os.listdir(SCAN) if f.lower().endswith((".jpg", ".jpeg", ".png"))}

def resolve(img_id):
    s = str(img_id).strip()
    if s.lower() in on_disk:
        return on_disk[s.lower()]
    stem = os.path.splitext(s)[0].lower()
    for ext in (".jpg", ".jpeg", ".png"):
        if stem + ext in on_disk:
            return on_disk[stem + ext]
    return None

meta["_file"] = meta["Image ID"].map(resolve)
missing = meta["_file"].isna().sum()
print(f"unmatched metadata rows: {missing}")
orphans = set(on_disk.values()) - set(meta["_file"].dropna())
print(f"images with no metadata row: {len(orphans)}")

# patient id from filename: MYS_MEM_UMIC0001_6_1.jpg -> UMIC0001
def patient_id(fn):
    m = re.match(r"([A-Z]{3})_([A-Z]+)_([A-Za-z0-9]+)_", str(fn))
    return m.group(3) if m else None

def country_code(fn):
    m = re.match(r"([A-Z]{3})_", str(fn))
    return m.group(1) if m else None

HIGH_RISK = {"Cancer", "OPMD"}

man = pd.DataFrame({
    "filename":   meta["_file"],
    "filepath":   meta["_file"].map(lambda f: os.path.join(SCAN, f) if pd.notna(f) else None),
    "site":       "SAMPLE_Malaysia",          # one collection = one site
    "country":    meta["_file"].map(country_code),
    "patient_id": meta["_file"].map(patient_id),
    "diagnosis":  meta["Lesion type"].astype(str).str.strip(),
    "clinical_dx": meta["Clinical diagnosis"].astype(str).str.strip(),
    "referral":   meta["Referral decision"].astype(str).str.strip(),
    "captured_by": meta["Image captured by"].astype(str).str.strip(),
    "age":        pd.to_numeric(meta["Age"], errors="coerce"),
    "gender":     meta["Gender"].astype(str).str.strip(),
    "smoking":    meta["Smoking"].astype(str).str.strip(),
    "betel":      meta["Chewing Betel Quid"].astype(str).str.strip(),
    "alcohol":    meta["Alcohol"].astype(str).str.strip(),
    "biopsy":     meta["Biopsy"].astype(str).str.strip(),
})
man["risk_label"] = man["diagnosis"].isin(HIGH_RISK).astype(int)
man = man[man.filepath.notna()].reset_index(drop=True)

os.makedirs(f"{ROOT}/outputs/tables", exist_ok=True)
man.to_csv(f"{ROOT}/outputs/tables/manifest_sample.csv", index=False)

print(f"\nmanifest: {len(man)} rows, {man.patient_id.nunique()} patients")
print(f"images per patient: min {man.groupby('patient_id').size().min()}, "
      f"max {man.groupby('patient_id').size().max()}")
print("\nclass balance:")
print(man.diagnosis.value_counts())
print(f"\nhigh-risk: {man.risk_label.sum()} / {len(man)} ({100*man.risk_label.mean():.0f}%)")
print("\nreferral x risk:")
print(pd.crosstab(man.referral, man.risk_label))
print("\ncaptured_by x risk:")
print(pd.crosstab(man.captured_by, man.risk_label))

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             precision_score, recall_score, confusion_matrix,
                             accuracy_score)

def score(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= thresh).astype(int)
    out = {
        "n": len(y_true),
        "pos_rate": round(float(y_true.mean()), 3),
        "acc":      round(accuracy_score(y_true, y_pred), 3),
        "f1_macro": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 3),
        "prec_hr":  round(precision_score(y_true, y_pred, zero_division=0), 3),
        "rec_hr":   round(recall_score(y_true, y_pred, zero_division=0), 3),
    }
    if len(np.unique(y_true)) > 1:
        out["auc"]    = round(roc_auc_score(y_true, y_prob), 3)
        out["pr_auc"] = round(average_precision_score(y_true, y_prob), 3)
    else:
        out["auc"] = out["pr_auc"] = None
    out["cm"] = confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist()
    return out


def within_site_cv(df, site, fit_predict, n_splits=5, seed=0):
    """Patient-grouped, stratified CV inside one site. No patient spans folds."""
    g = df[df.site == site].reset_index(drop=True)
    y, groups = g.risk_label.values, g.patient_id.values
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = []
    for tr, te in sgkf.split(g, y, groups):
        assert not (set(groups[tr]) & set(groups[te])), "patient leaked across folds"
        folds.append(score(y[te], fit_predict(g.iloc[tr], g.iloc[te])))
    agg = {}
    for k in ("acc", "f1_macro", "prec_hr", "rec_hr", "auc", "pr_auc"):
        vals = [f[k] for f in folds if f[k] is not None]
        agg[k] = round(float(np.mean(vals)), 3) if vals else None
    agg.update(eval_type="within", site=site, n=len(g),
               n_patients=g.patient_id.nunique())
    return agg, folds


def leave_one_site_out(df, train_sites, test_site, fit_predict):
    tr = df[df.site.isin(train_sites)].reset_index(drop=True)
    te = df[df.site == test_site].reset_index(drop=True)
    assert not (set(tr.patient_id) & set(te.patient_id)), "patient in both sites"
    r = score(te.risk_label.values, fit_predict(tr, te))
    r.update(eval_type="loso", train_sites="+".join(train_sites), site=test_site)
    return r


# ---------- sanity check: random predictor ----------
def fit_predict_random(train_df, test_df):
    return np.random.default_rng(0).random(len(test_df))

man = pd.read_csv(f"{ROOT}/outputs/tables/manifest_sample.csv")
agg, folds = within_site_cv(man, "SAMPLE_Malaysia", fit_predict_random)
print("random predictor, within-site:")
print(" ", agg)
print("\nper-fold test sizes:", [f["n"] for f in folds])
print("per-fold pos rates :", [f["pos_rate"] for f in folds])

In [ ]:
import numpy as np, pandas as pd
from PIL import Image
from skimage.feature import local_binary_pattern
from skimage.color import rgb2hsv
from tqdm import tqdm

def image_features(path, size=256):
    im = Image.open(path).convert("RGB").resize((size, size))
    a = np.asarray(im, dtype=np.float32) / 255.0

    feats = []
    # per-channel stats
    for c in range(3):
        ch = a[:, :, c]
        feats += [ch.mean(), ch.std(),
                  np.percentile(ch, 25), np.percentile(ch, 75)]

    # RGB histograms, 16 bins each
    for c in range(3):
        h, _ = np.histogram(a[:, :, c], bins=16, range=(0, 1), density=True)
        feats += list(h)

    # HSV histograms — hue matters for mucosa vs lesion colour
    hsv = rgb2hsv(a)
    for c in range(3):
        h, _ = np.histogram(hsv[:, :, c], bins=16, range=(0, 1), density=True)
        feats += list(h)

    # LBP texture on greyscale
    grey = (a @ [0.299, 0.587, 0.114])
    lbp = local_binary_pattern((grey * 255).astype(np.uint8), P=8, R=1, method="uniform")
    h, _ = np.histogram(lbp, bins=10, range=(0, 10), density=True)
    feats += list(h)

    return np.array(feats, dtype=np.float32)

FEAT_CACHE = {}
for p in tqdm(man.filepath.tolist(), desc="features"):
    FEAT_CACHE[p] = image_features(p)

FEAT_DIM = len(next(iter(FEAT_CACHE.values())))
print("feature dim:", FEAT_DIM)

In [ ]:
rows = []
for name, fn in [("majority", fit_predict_majority),
                 ("random",   fit_predict_random),
                 ("rf_colour_texture", fit_predict_rf)]:
    agg, _ = within_site_cv(man, "SAMPLE_Malaysia", fn)
    agg["model"] = name
    rows.append(agg)

res = pd.DataFrame(rows)[["model", "n", "n_patients", "auc", "pr_auc",
                          "f1_macro", "acc", "prec_hr", "rec_hr"]]
res.to_csv(f"{ROOT}/outputs/tables/baselines_sample.csv", index=False)
print(res.to_string(index=False))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def X_of(df):
    return np.stack([FEAT_CACHE[p] for p in df.filepath])

def fit_predict_majority(train_df, test_df):
    """Always predicts the training set's base rate. The floor."""
    return np.full(len(test_df), train_df.risk_label.mean())

def fit_predict_rf(train_df, test_df, seed=0):
    clf = make_pipeline(
        StandardScaler(),
        RandomForestClassifier(n_estimators=400, min_samples_leaf=2,
                               class_weight="balanced_subsample",
                               random_state=seed, n_jobs=-1),
    )
    clf.fit(X_of(train_df), train_df.risk_label.values)
    return clf.predict_proba(X_of(test_df))[:, 1]

print("defined:", fit_predict_majority, fit_predict_rf)

In [ ]:
rows = []
for name, fn in [("majority", fit_predict_majority),
                 ("random",   fit_predict_random),
                 ("rf_colour_texture", fit_predict_rf)]:
    agg, _ = within_site_cv(man, "SAMPLE_Malaysia", fn)
    agg["model"] = name
    rows.append(agg)

res = pd.DataFrame(rows)[["model", "n", "n_patients", "auc", "pr_auc",
                          "f1_macro", "acc", "prec_hr", "rec_hr"]]
res.to_csv(f"{ROOT}/outputs/tables/baselines_sample.csv", index=False)
print(res.to_string(index=False))

In [ ]:
import torch, torch.nn as nn, numpy as np
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG = 224

# colour/brightness jitter on purpose — it's the RQ2 hypothesis made operational
train_tf = A.Compose([
    A.RandomResizedCrop(size=(IMG, IMG), scale=(0.7, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05, p=0.8),
    A.Normalize(), ToTensorV2(),
])
eval_tf = A.Compose([A.Resize(IMG, IMG), A.Normalize(), ToTensorV2()])

class OralDS(Dataset):
    def __init__(self, df, tf):
        self.paths = df.filepath.tolist()
        self.y = df.risk_label.values.astype(np.float32)
        self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = np.array(Image.open(self.paths[i]).convert("RGB"))
        return self.tf(image=img)["image"], self.y[i]

def fit_predict_cnn(train_df, test_df, arch="resnet18", epochs=8,
                    bs=16, lr=3e-4, seed=0, verbose=False):
    torch.manual_seed(seed)
    model = timm.create_model(arch, pretrained=True, num_classes=1).to(DEVICE)

    pos_w = torch.tensor([(1 - train_df.risk_label.mean()) /
                          max(train_df.risk_label.mean(), 1e-6)], device=DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    tl = DataLoader(OralDS(train_df, train_tf), batch_size=bs, shuffle=True,
                    num_workers=2, drop_last=False)
    el = DataLoader(OralDS(test_df, eval_tf), batch_size=bs, shuffle=False,
                    num_workers=2)

    model.train()
    for ep in range(epochs):
        tot = 0.0
        for x, y in tl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(x).squeeze(1), y)
            loss.backward(); opt.step()
            tot += loss.item() * len(y)
        if verbose:
            print(f"  epoch {ep+1}: loss {tot/len(train_df):.4f}")

    model.eval()
    probs = []
    with torch.no_grad():
        for x, _ in el:
            probs.append(torch.sigmoid(model(x.to(DEVICE)).squeeze(1)).cpu().numpy())
    return np.concatenate(probs)

In [ ]:
agg, folds = within_site_cv(man, "SAMPLE_Malaysia", fit_predict_cnn)
agg["model"] = "resnet18_finetuned"
print(agg)

res2 = pd.concat([res, pd.DataFrame([agg])[res.columns]], ignore_index=True)
res2.to_csv(f"{ROOT}/outputs/tables/baselines_sample.csv", index=False)
print("\n" + res2.to_string(index=False))

In [ ]:
man2 = man.copy()
man2["site"] = "cap_" + man2.captured_by.str.replace(" ", "_")

print(man2.groupby("site").agg(n=("filename","size"),
                               patients=("patient_id","nunique"),
                               pos_rate=("risk_label","mean")))

# do any patients appear under both photographers?
overlap = (man2.groupby("patient_id").site.nunique() > 1).sum()
print(f"\npatients photographed by both: {overlap}")

In [ ]:
# find and remove the patient who appears under both photographers
both = man2.groupby("patient_id").site.nunique()
shared = both[both > 1].index.tolist()
print("dropping patient(s):", shared)

man2c = man2[~man2.patient_id.isin(shared)].reset_index(drop=True)
print(man2c.groupby("site").agg(n=("filename","size"),
                                patients=("patient_id","nunique"),
                                pos_rate=("risk_label","mean")))

# LOSO both directions, RF and CNN
loso_rows = []
for model_name, fn in [("rf_colour_texture", fit_predict_rf),
                       ("resnet18_finetuned", fit_predict_cnn)]:
    for tr, te in [(["cap_Clinician"], "cap_Study_Coordinator"),
                   (["cap_Study_Coordinator"], "cap_Clinician")]:
        r = leave_one_site_out(man2c, tr, te, fn)
        r["model"] = model_name
        loso_rows.append(r)

loso = pd.DataFrame(loso_rows)[["model", "train_sites", "site", "n",
                                "pos_rate", "auc", "pr_auc", "f1_macro",
                                "acc", "prec_hr", "rec_hr"]]
loso.to_csv(f"{ROOT}/outputs/tables/loso_photographer_sample.csv", index=False)
print("\n" + loso.to_string(index=False))


In [ ]:
%%writefile {ROOT}/src/pipeline.py
# paste the contents of: score, within_site_cv, leave_one_site_out,
# image_features, X_of, fit_predict_majority, fit_predict_rf,
# OralDS, train_tf, eval_tf, fit_predict_cnn

In [ ]:
%%writefile /content/drive/MyDrive/oral_cancer_crosssite/src/pipeline.py
"""Cross-site oral lesion classification — shared pipeline."""
import os, numpy as np, pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             precision_score, recall_score, confusion_matrix,
                             accuracy_score)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from skimage.feature import local_binary_pattern
from skimage.color import rgb2hsv

HIGH_RISK = {"Cancer", "OPMD"}

# ---------------- metrics ----------------

def score(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= thresh).astype(int)
    out = {
        "n": len(y_true),
        "pos_rate": round(float(y_true.mean()), 3),
        "acc":      round(accuracy_score(y_true, y_pred), 3),
        "f1_macro": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 3),
        "prec_hr":  round(precision_score(y_true, y_pred, zero_division=0), 3),
        "rec_hr":   round(recall_score(y_true, y_pred, zero_division=0), 3),
    }
    if len(np.unique(y_true)) > 1:
        out["auc"]    = round(roc_auc_score(y_true, y_prob), 3)
        out["pr_auc"] = round(average_precision_score(y_true, y_prob), 3)
    else:
        out["auc"] = out["pr_auc"] = None
    out["cm"] = confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist()
    return out

# ---------------- evaluation ----------------

def within_site_cv(df, site, fit_predict, n_splits=5, seed=0):
    g = df[df.site == site].reset_index(drop=True)
    y, groups = g.risk_label.values, g.patient_id.values
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = []
    for tr, te in sgkf.split(g, y, groups):
        assert not (set(groups[tr]) & set(groups[te])), "patient leaked across folds"
        folds.append(score(y[te], fit_predict(g.iloc[tr], g.iloc[te])))
    agg = {}
    for k in ("acc", "f1_macro", "prec_hr", "rec_hr", "auc", "pr_auc"):
        vals = [f[k] for f in folds if f[k] is not None]
        agg[k] = round(float(np.mean(vals)), 3) if vals else None
    agg.update(eval_type="within", site=site,

In [ ]:
%%writefile /content/drive/MyDrive/oral_cancer_crosssite/src/pipeline.py
"""Cross-site oral lesion classification — shared pipeline."""
import os, numpy as np, pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             precision_score, recall_score, confusion_matrix,
                             accuracy_score)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from skimage.feature import local_binary_pattern
from skimage.color import rgb2hsv

HIGH_RISK = {"Cancer", "OPMD"}

# ---------------- metrics ----------------

def score(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= thresh).astype(int)
    out = {
        "n": len(y_true),
        "pos_rate": round(float(y_true.mean()), 3),
        "acc":      round(accuracy_score(y_true, y_pred), 3),
        "f1_macro": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 3),
        "prec_hr":  round(precision_score(y_true, y_pred, zero_division=0), 3),
        "rec_hr":   round(recall_score(y_true, y_pred, zero_division=0), 3),
    }
    if len(np.unique(y_true)) > 1:
        out["auc"]    = round(roc_auc_score(y_true, y_prob), 3)
        out["pr_auc"] = round(average_precision_score(y_true, y_prob), 3)
    else:
        out["auc"] = out["pr_auc"] = None
    out["cm"] = confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist()
    return out

# ---------------- evaluation ----------------

def within_site_cv(df, site, fit_predict, n_splits=5, seed=0):
    g = df[df.site == site].reset_index(drop=True)
    y, groups = g.risk_label.values, g.patient_id.values
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = []
    for tr, te in sgkf.split(g, y, groups):
        assert not (set(groups[tr]) & set(groups[te])), "patient leaked across folds"
        folds.append(score(y[te], fit_predict(g.iloc[tr], g.iloc[te])))
    agg = {}
    for k in ("acc", "f1_macro", "prec_hr", "rec_hr", "auc", "pr_auc"):
        vals = [f[k] for f in folds if f[k] is not None]
        agg[k] = round(float(np.mean(vals)), 3) if vals else None
    agg.update(eval_type="within", site=site, n=len(g),
               n_patients=g.patient_id.nunique())
    return agg, folds


def leave_one_site_out(df, train_sites, test_site, fit_predict):
    tr = df[df.site.isin(train_sites)].reset_index(drop=True)
    te = df[df.site == test_site].reset_index(drop=True)
    assert not (set(tr.patient_id) & set(te.patient_id)), "patient in both sites"
    r = score(te.risk_label.values, fit_predict(tr, te))
    r.update(eval_type="loso", train_sites="+".join(train_sites), site=test_site)
    return r

# ---------------- handcrafted features ----------------

def image_features(path, size=256):
    im = Image.open(path).convert("RGB").resize((size, size))
    a = np.asarray(im, dtype=np.float32) / 255.0
    feats = []
    for c in range(3):
        ch = a[:, :, c]
        feats += [ch.mean(), ch.std(), np.percentile(ch, 25), np.percentile(ch, 75)]
    for c in range(3):
        h, _ = np.histogram(a[:, :, c], bins=16, range=(0, 1), density=True)
        feats += list(h)
    hsv = rgb2hsv(a)
    for c in range(3):
        h, _ = np.histogram(hsv[:, :, c], bins=16, range=(0, 1), density=True)
        feats += list(h)
    grey = a @ [0.299, 0.587, 0.114]
    lbp = local_binary_pattern((grey * 255).astype(np.uint8), P=8, R=1, method="uniform")
    h, _ = np.histogram(lbp, bins=10, range=(0, 10), density=True)
    feats += list(h)
    return np.array(feats, dtype=np.float32)


def build_feature_cache(paths, cache=None):
    cache = {} if cache is None else cache
    for p in paths:
        if p not in cache:
            cache[p] = image_features(p)
    return cache

# ---------------- baseline models ----------------

def make_X(df, cache):
    return np.stack([cache[p] for p in df.filepath])


def fit_predict_majority(train_df, test_df):
    return np.full(len(test_df), train_df.risk_label.mean())


def fit_predict_random(train_df, test_df, seed=0):
    return np.random.default_rng(seed).random(len(test_df))


def make_rf_predictor(cache, seed=0):
    def f(train_df, test_df):
        clf = make_pipeline(
            StandardScaler(),
            RandomForestClassifier(n_estimators=400, min_samples_leaf=2,
                                   class_weight="balanced_subsample",
                                   random_state=seed, n_jobs=-1),
        )
        clf.fit(make_X(train_df, cache), train_df.risk_label.values)
        return clf.predict_proba(make_X(test_df, cache))[:, 1]
    return f

# ---------------- CNN ----------------

def make_cnn_predictor(arch="resnet18", epochs=8, bs=16, lr=3e-4,
                       img=224, seed=0, verbose=False):
    import torch, torch.nn as nn, timm
    from torch.utils.data import Dataset, DataLoader
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    train_tf = A.Compose([
        A.RandomResizedCrop(size=(img, img), scale=(0.7, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05, p=0.8),
        A.Normalize(), ToTensorV2(),
    ])
    eval_tf = A.Compose([A.Resize(img, img), A.Normalize(), ToTensorV2()])

    class OralDS(Dataset):
        def __init__(self, df, tf):
            self.paths = df.filepath.tolist()
            self.y = df.risk_label.values.astype(np.float32)
            self.tf = tf
        def __len__(self): return len(self.paths)
        def __getitem__(self, i):
            im = np.array(Image.open(self.paths[i]).convert("RGB"))
            return self.tf(image=im)["image"], self.y[i]

    def f(train_df, test_df):
        torch.manual_seed(seed)
        model = timm.create_model(arch, pretrained=True, num_classes=1).to(DEVICE)
        rate = max(train_df.risk_label.mean(), 1e-6)
        crit = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([(1 - rate) / rate], device=DEVICE))
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        tl = DataLoader(OralDS(train_df, train_tf), batch_size=bs,
                        shuffle=True, num_workers=2)
        el = DataLoader(OralDS(test_df, eval_tf), batch_size=bs,
                        shuffle=False, num_workers=2)
        model.train()
        for ep in range(epochs):
            tot = 0.0
            for x, y in tl:
                x, y = x.to(DEVICE), y.to(DEVICE)
                opt.zero_grad()
                loss = crit(model(x).squeeze(1), y)
                loss.backward(); opt.step()
                tot += loss.item() * len(y)
            if verbose:
                print(f"  epoch {ep+1}: loss {tot/len(train_df):.4f}")
        model.eval()
        probs = []
        with torch.no_grad():
            for x, _ in el:
                probs.append(torch.sigmoid(model(x.to(DEVICE)).squeeze(1)).cpu().numpy())
        return np.concatenate(probs)
    return f

In [ ]:
print(open(f"{ROOT}/src/pipeline.py").read()[:400])

In [ ]:
import pandas as pd, json

SL = f"{ROOT}/data/raw/srilanka"

img = pd.read_csv(f"{SL}/Imagewise_Data.csv")
pat = pd.read_csv(f"{SL}/Patientwise_Data.csv")

print("=== Imagewise_Data.csv ===")
print("shape:", img.shape)
print("columns:", img.columns.tolist())
print(img.head(3).to_string())
for c in img.columns:
    if img[c].nunique(dropna=True) <= 12:
        print(f"\n-- {c} --")
        print(img[c].value_counts(dropna=False))

print("\n\n=== Patientwise_Data.csv ===")
print("shape:", pat.shape)
print("columns:", pat.columns.tolist())
print(pat.head(3).to_string())
for c in pat.columns:
    if pat[c].nunique(dropna=True) <= 12:
        print(f"\n-- {c} --")
        print(pat[c].value_counts(dropna=False))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
ROOT = "/content/drive/MyDrive/oral_cancer_crosssite"
sys.path.append(f"{ROOT}/src")
os.makedirs(f"{ROOT}/data/raw/srilanka", exist_ok=True)

!pip install -q timm albumentations

from pipeline import *     # everything we saved yesterday
print("ready:", ROOT)

In [ ]:
import pandas as pd

SL = f"{ROOT}/data/raw/srilanka"
print("files present:", os.listdir(SL))

img = pd.read_csv(f"{SL}/Imagewise_Data.csv")
pat = pd.read_csv(f"{SL}/Patientwise_Data.csv")

print("\n=== Imagewise_Data.csv ===")
print("shape:", img.shape)
print("columns:", img.columns.tolist())
print(img.head(3).to_string())
for c in img.columns:
    if img[c].nunique(dropna=True) <= 12:
        print(f"\n-- {c} --")
        print(img[c].value_counts(dropna=False))

print("\n\n=== Patientwise_Data.csv ===")
print("shape:", pat.shape)
print("columns:", pat.columns.tolist())
print(pat.head(3).to_string())
for c in pat.columns:
    if pat[c].nunique(dropna=True) <= 12:
        print(f"\n-- {c} --")
        print(pat[c].value_counts(dropna=False))

In [ ]:
import re

HIGH_RISK_SL = {"OPMD", "OCA"}

sl = pd.DataFrame({
    "filename":   img["Image Name"].astype(str).str.strip(),
    "site":       "SriLanka_Peradeniya",
    "country":    "LKA",
    "patient_id": img["Image Name"].astype(str).str.extract(r"^(R-\d+)")[0],
    "diagnosis":  img["Category"].astype(str).str.strip(),
    "clinical_dx": img["Clinical Diagnosis"].astype(str).str.strip(),
    "n_lesion_ann": img["Lesion Annotation Count"],
})
sl["risk_label"] = sl["diagnosis"].isin(HIGH_RISK_SL).astype(int)

# merge patient-level metadata
pat2 = pat.rename(columns={
    "Patient ID": "patient_id", "Age": "age", "Gender": "gender",
    "Smoking": "smoking", "Chewing_Betel_Quid ": "betel", "Alcohol": "alcohol",
})[["patient_id", "age", "gender", "smoking", "betel", "alcohol"]]
sl = sl.merge(pat2, on="patient_id", how="left")

print("unmatched patient ids:", sl.patient_id.isna().sum())
print("rows with no metadata :", sl.age.isna().sum())
print(f"\n{len(sl)} images, {sl.patient_id.nunique()} patients")
print(f"images/patient: min {sl.groupby('patient_id').size().min()}, "
      f"max {sl.groupby('patient_id').size().max()}")
print("\nclass balance:")
print(sl.diagnosis.value_counts())
print(f"high-risk: {sl.risk_label.sum()}/{len(sl)} ({100*sl.risk_label.mean():.0f}%)")
print("\ntop clinical diagnoses:")
print(sl.clinical_dx.value_counts().head(15))

sl.to_csv(f"{ROOT}/outputs/tables/manifest_srilanka_nopaths.csv", index=False)

In [ ]:
names = img["Image Name"].astype(str).str.strip()

# what do the ID prefixes actually look like?
print("=== sample image names ===")
print(names.sample(20, random_state=0).tolist())

print("\n=== patient-id prefix patterns ===")
# strip the trailing image number, whatever the format
stem = names.str.replace(r"[-_]\d+$", "", regex=True)
print("unique stems:", stem.nunique())
print(stem.head(10).tolist())

print("\n=== leading letter groups ===")
print(names.str.extract(r"^([A-Za-z]+)")[0].value_counts())

print("\n=== patient ids in Patientwise ===")
print(pat["Patient ID"].astype(str).head(10).tolist())
print("prefixes:", pat["Patient ID"].astype(str).str.extract(r"^([A-Za-z]+)")[0].value_counts().to_dict())

In [ ]:
# general parser: everything before the final -NN is the patient id
sl = pd.DataFrame({
    "filename":    names,
    "site":        "SriLanka_Peradeniya",
    "country":     "LKA",
    "patient_id":  names.str.replace(r"[-_]\d+$", "", regex=True),
    "subsite":     names.str.extract(r"^([A-Za-z]+)")[0],
    "diagnosis":   img["Category"].astype(str).str.strip(),
    "clinical_dx": img["Clinical Diagnosis"].astype(str).str.strip(),
    "n_lesion_ann": img["Lesion Annotation Count"],
})
sl["risk_label"] = sl["diagnosis"].isin({"OPMD", "OCA"}).astype(int)

pat2 = pat.rename(columns={
    "Patient ID": "patient_id", "Age": "age", "Gender": "gender",
    "Smoking": "smoking", "Chewing_Betel_Quid ": "betel", "Alcohol": "alcohol",
})[["patient_id", "age", "gender", "smoking", "betel", "alcohol"]]
pat2["patient_id"] = pat2.patient_id.astype(str).str.strip()

sl = sl.merge(pat2, on="patient_id", how="left")

print("images:", len(sl), " patients in images:", sl.patient_id.nunique(),
      " patients in metadata:", pat2.patient_id.nunique())
print("rows missing metadata:", sl.age.isna().sum())

missing = sorted(set(sl.patient_id) - set(pat2.patient_id))
print(f"\npatient ids in images but NOT in Patientwise: {len(missing)}")
print("examples:", missing[:15])

print("\n=== per sub-site breakdown ===")
print(sl.groupby("subsite").agg(
    images=("filename", "size"),
    patients=("patient_id", "nunique"),
    high_risk_rate=("risk_label", "mean"),
    missing_meta=("age", lambda s: s.isna().sum()),
).round(3))

print("\n=== diagnosis mix by sub-site ===")
print(pd.crosstab(sl.subsite, sl.diagnosis))

print("\n=== top clinical dx by sub-site ===")
for s in sorted(sl.subsite.unique()):
    top = sl[sl.subsite == s].clinical_dx.value_counts().head(5)
    print(f"\n{s}:", dict(top))

In [ ]:
C = sl[sl.subsite == "C"]
print("C filenames:", C.filename.head(12).tolist())
print("C parsed patient_ids:", C.patient_id.head(12).tolist())
print("\nnon-C filename samples:", sl[sl.subsite != "C"].filename.head(6).tolist())

# how many parts in the name, by subsite
sl["n_parts"] = sl.filename.str.count("-") + 1
print("\nname parts by subsite:")
print(pd.crosstab(sl.subsite, sl.n_parts))

In [ ]:
def parse_patient(fn):
    """C-10-12-5 -> C-10 (4 parts). R-01-01 -> R-01 (3 parts)."""
    parts = fn.split("-")
    return "-".join(parts[:2]) if len(parts) >= 4 else "-".join(parts[:2])

sl["patient_id"] = sl.filename.map(parse_patient)

# re-merge metadata with corrected ids
sl = sl.drop(columns=["age", "gender", "smoking", "betel", "alcohol"], errors="ignore")
sl = sl.merge(pat2, on="patient_id", how="left")

print("images:", len(sl), " patients:", sl.patient_id.nunique())
print("rows missing metadata:", sl.age.isna().sum())
print("\nper sub-site:")
print(sl.groupby("subsite").agg(
    images=("filename", "size"),
    patients=("patient_id", "nunique"),
    high_risk=("risk_label", "mean"),
    no_meta=("age", lambda s: s.isna().sum()),
).round(3))

# patients must not span sub-sites
span = sl.groupby("patient_id").subsite.nunique()
print("\npatients spanning sub-sites:", (span > 1).sum())

sl.to_csv(f"{ROOT}/outputs/tables/manifest_srilanka_nopaths.csv", index=False)
print("\nsaved")

In [ ]:
from getpass import getpass
TOKEN = getpass("Zenodo token: ")

url = f"https://zenodo.org/api/records/10664056/files/Images.zip/content?access_token={TOKEN}"
!wget -q --show-progress "{url}" -O /content/Images.zip
!ls -lh /content/Images.zip

In [ ]:
import os, time
t0 = time.time()
os.makedirs("/content/srilanka", exist_ok=True)
!unzip -q /content/Images.zip -d /content/srilanka
print(f"extracted in {time.time()-t0:.0f}s")

# what came out?
for dirpath, dirnames, filenames in os.walk("/content/srilanka"):
    rel = os.path.relpath(dirpath, "/content/srilanka")
    imgs = [f for f in filenames if f.lower().endswith((".jpg",".jpeg",".png"))]
    if imgs or dirnames:
        print(f"{len(imgs):6d} images  {rel}")
        if imgs:
            print(f"         e.g. {imgs[:3]}")

In [ ]:
import glob, zipfile, shutil, pandas as pd, numpy as np

# ---- Sri Lanka paths ----
SL_IMG = "/content/srilanka/Images"
sl = pd.read_csv(f"{ROOT}/outputs/tables/manifest_srilanka_nopaths.csv")
sl["filepath"] = SL_IMG + "/" + sl.filename + ".jpg"
missing = ~sl.filepath.map(os.path.exists)
print(f"Sri Lanka: {len(sl)} rows, {missing.sum()} missing files")

# ---- Malaysia sample: re-extract ----
zips = glob.glob(f"{ROOT}/data/sample/*.zip")
shutil.rmtree("/content/sample", ignore_errors=True)
os.makedirs("/content/sample", exist_ok=True)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall("/content/sample")

mys = pd.read_csv(f"{ROOT}/outputs/tables/manifest_sample.csv")
mys["filepath"] = mys.filename.map(lambda f: f"/content/sample/{f}")
print(f"Malaysia : {len(mys)} rows, {(~mys.filepath.map(os.path.exists)).sum()} missing files")

# ---- merge into one manifest; Sri Lanka sub-sites become sites ----
sl["site"] = "LKA_" + sl.subsite
mys["site"] = "MYS_sample"

cols = ["filepath","filename","site","country","patient_id","diagnosis",
        "clinical_dx","risk_label","age","gender","smoking","betel","alcohol"]
for c in cols:
    for d in (sl, mys):
        if c not in d.columns: d[c] = np.nan

man_all = pd.concat([sl[cols], mys[cols]], ignore_index=True)
man_all = man_all[man_all.filepath.map(os.path.exists)].reset_index(drop=True)
man_all.to_csv(f"{ROOT}/outputs/tables/manifest_all.csv", index=False)

print(f"\n=== merged manifest: {len(man_all)} images ===")
prof = man_all.groupby("site").agg(
    images=("filename","size"),
    patients=("patient_id","nunique"),
    high_risk=("risk_label","mean"),
    mean_age=("age","mean"),
    betel_yes=("betel", lambda s: (s.astype(str).str.strip().str.lower()=="yes").mean()),
).round(3)
print(prof)
print("\npatients spanning sites:", (man_all.groupby("patient_id").site.nunique()>1).sum())

In [ ]:
from PIL import Image
rows = []
for site, g in man_all.groupby("site"):
    s = g.sample(min(200, len(g)), random_state=0)
    w,h,kb,bright = [],[],[],[]
    for p in s.filepath:
        try:
            with Image.open(p) as im:
                im = im.convert("RGB"); w.append(im.size[0]); h.append(im.size[1])
                bright.append(np.asarray(im.resize((64,64))).mean())
            kb.append(os.path.getsize(p)/1024)
        except Exception: pass
    rows.append(dict(site=site, n=len(g),
                     med_w=int(np.median(w)), med_h=int(np.median(h)),
                     med_kb=round(np.median(kb),1),
                     brightness=round(np.mean(bright),1)))
prof_img = pd.DataFrame(rows)
prof_img.to_csv(f"{ROOT}/outputs/tables/site_profile.csv", index=False)
print(prof_img.to_string(index=False))

In [ ]:
from PIL import Image
from tqdm import tqdm
import os

CACHE = "/content/img512"
os.makedirs(CACHE, exist_ok=True)

def shrink(src, dst, max_side=512):
    if os.path.exists(dst): return
    with Image.open(src) as im:
        im = im.convert("RGB")
        s = max_side / max(im.size)
        if s < 1:
            im = im.resize((round(im.size[0]*s), round(im.size[1]*s)), Image.LANCZOS)
        im.save(dst, "JPEG", quality=92)

man_all = pd.read_csv(f"{ROOT}/outputs/tables/manifest_all.csv")
man_all["path512"] = [f"{CACHE}/{s}__{os.path.basename(p)}"
                      for s, p in zip(man_all.site, man_all.filepath)]

for src, dst in tqdm(list(zip(man_all.filepath, man_all.path512)), desc="resize"):
    shrink(src, dst)

man_all["filepath_full"] = man_all.filepath
man_all["filepath"] = man_all.path512          # everything downstream uses 512px
man_all.to_csv(f"{ROOT}/outputs/tables/manifest_all.csv", index=False)

import subprocess
print("\ncache size:", subprocess.run(["du","-sh",CACHE], capture_output=True, text=True).stdout)
print("missing:", (~man_all.filepath.map(os.path.exists)).sum())

In [ ]:
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True   # salvage partial JPEGs

bad = []
for src, dst in tqdm(list(zip(man_all.filepath, man_all.path512)), desc="resize"):
    try:
        shrink(src, dst)
    except Exception as e:
        bad.append((src, type(e).__name__, str(e)[:60]))

print(f"\nfailed: {len(bad)}")
for b in bad: print("  ", b)

ok = man_all.path512.map(os.path.exists)
print(f"cached: {ok.sum()}/{len(man_all)}")

man_all["filepath_full"] = man_all.filepath
man_all["filepath"] = man_all.path512
man_all = man_all[ok].reset_index(drop=True)
man_all.to_csv(f"{ROOT}/outputs/tables/manifest_all.csv", index=False)

print("\nfinal manifest:")
print(man_all.groupby("site").agg(images=("filename","size"),
                                  patients=("patient_id","nunique"),
                                  high_risk=("risk_label","mean")).round(3))

In [ ]:
import time
t0 = time.time()
cache = build_feature_cache(man_all.filepath.tolist())
print(f"features for {len(cache)} images in {time.time()-t0:.0f}s")

rf  = make_rf_predictor(cache)
cnn = make_cnn_predictor(arch="resnet18", epochs=8, bs=32, lr=3e-4)

In [ ]:
SITE = "LKA_R"
rows = []
for name, fn in [("majority", fit_predict_majority),
                 ("rf_colour_texture", rf),
                 ("resnet18", cnn)]:
    t = time.time()
    agg, folds = within_site_cv(man_all, SITE, fn)
    agg["model"] = name
    agg["secs"] = round(time.time() - t)
    rows.append(agg)
    print(f"{name:20s} auc={agg['auc']}  f1={agg['f1_macro']}  ({agg['secs']}s)")

rq1 = pd.DataFrame(rows)[["model","site","n","n_patients","auc","pr_auc",
                          "f1_macro","acc","prec_hr","rec_hr","secs"]]
rq1.to_csv(f"{ROOT}/outputs/tables/rq1_within_site_R.csv", index=False)
print("\n" + rq1.to_string(index=False))

In [ ]:
TRAIN_SITE = "LKA_R"
TEST_SITES = ["LKA_N", "LKA_S", "MYS_sample"]

rows = []

# within-site reference for each test site
for s in TEST_SITES:
    for name, fn in [("rf_colour_texture", rf), ("resnet18", cnn)]:
        agg, _ = within_site_cv(man_all, s, fn)
        rows.append(dict(model=name, eval="within", train=s, test=s,
                         n=agg["n"], auc=agg["auc"], pr_auc=agg["pr_auc"],
                         f1_macro=agg["f1_macro"], rec_hr=agg["rec_hr"]))
        print(f"within {s:12s} {name:18s} auc={agg['auc']}")

# cross-site: train on R, test elsewhere
for s in TEST_SITES:
    for name, fn in [("rf_colour_texture", rf), ("resnet18", cnn)]:
        r = leave_one_site_out(man_all, [TRAIN_SITE], s, fn)
        rows.append(dict(model=name, eval="cross", train=TRAIN_SITE, test=s,
                         n=r["n"], auc=r["auc"], pr_auc=r["pr_auc"],
                         f1_macro=r["f1_macro"], rec_hr=r["rec_hr"]))
        print(f"cross  {TRAIN_SITE}->{s:12s} {name:18s} auc={r['auc']}")

rq2 = pd.DataFrame(rows)
rq2.to_csv(f"{ROOT}/outputs/tables/rq2_loso_raw.csv", index=False)

# the headline table: within vs cross, and the drop
piv = rq2.pivot_table(index=["model","test"], columns="eval",
                      values="auc").reset_index()
piv["drop"] = (piv["within"] - piv["cross"]).round(3)
piv["pct_retained"] = (100 * piv["cross"] / piv["within"]).round(1)
piv.to_csv(f"{ROOT}/outputs/tables/rq2_headline.csv", index=False)
print("\n=== RQ2: trained on LKA_R ===")
print(piv.to_string(index=False))

In [ ]:
from itertools import product

SITES = ["LKA_R", "LKA_N", "LKA_S", "MYS_sample"]
N_TRAIN = 640          # fixed training budget, so size can't explain the gap

def boot_auc(y, p, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p)
    out = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) > 1:
            out.append(roc_auc_score(y[i], p[i]))
    return (round(float(np.percentile(out, 2.5)), 3),
            round(float(np.percentile(out, 97.5)), 3))

def subsample_patients(df, n_img, seed=0):
    """Trim to ~n_img images without splitting patients."""
    if len(df) <= n_img: return df
    rng = np.random.default_rng(seed)
    pids = rng.permutation(df.patient_id.unique())
    keep, tot = [], 0
    for p in pids:
        k = (df.patient_id == p).sum()
        if tot + k > n_img: break
        keep.append(p); tot += k
    return df[df.patient_id.isin(keep)].reset_index(drop=True)

rows = []
for tr_site, te_site in product(SITES, SITES):
    if tr_site == te_site: continue
    tr = subsample_patients(man_all[man_all.site == tr_site], N_TRAIN)
    te = man_all[man_all.site == te_site].reset_index(drop=True)
    if tr.risk_label.nunique() < 2 or len(tr) < 100: continue
    p = cnn(tr, te)
    r = score(te.risk_label.values, p)
    lo, hi = boot_auc(te.risk_label.values, p)
    rows.append(dict(train=tr_site, test=te_site, n_train=len(tr), n_test=len(te),
                     auc=r["auc"], ci_lo=lo, ci_hi=hi,
                     f1=r["f1_macro"], rec_hr=r["rec_hr"],
                     cross_country=(tr_site[:3] != te_site[:3])))
    print(f"{tr_site:12s} -> {te_site:12s}  auc={r['auc']}  [{lo}, {hi}]")

mat = pd.DataFrame(rows)
mat.to_csv(f"{ROOT}/outputs/tables/rq2_matrix_fixed_budget.csv", index=False)

print("\n=== within-country vs cross-country ===")
print(mat.groupby("cross_country").auc.agg(["mean","min","max","count"]).round(3))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mat = pd.read_csv(f"{ROOT}/outputs/tables/rq2_matrix_fixed_budget.csv")

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
mpl.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10})

d = mat.copy()
d["label"] = d.train.str.replace("_sample","") + " \u2192 " + d.test.str.replace("_sample","")
d["grp"] = np.where(d.cross_country, "Cross-country", "Within-country")
d = d.sort_values(["cross_country","auc"], ascending=[True, True]).reset_index(drop=True)

# gap between the two groups
ypos, y = [], 0.0
prev = d.grp.iloc[0]
for g in d.grp:
    if g != prev: y += 1.2; prev = g
    ypos.append(y); y += 1.0
d["y"] = ypos

fig, ax = plt.subplots(figsize=(7.6, 5.4), dpi=160)
ax.set_facecolor("#fcfcfb"); fig.patch.set_facecolor("#fcfcfb")

ax.axvline(0.5, color=MUTED, ls="--", lw=1, zorder=1)
ax.text(0.5, d.y.max()+1.25, "chance", color=MUTED, fontsize=9,
        ha="center", va="bottom")

for g, c in [("Within-country", BLUE), ("Cross-country", ORANGE)]:
    s = d[d.grp == g]
    ax.hlines(s.y, s.ci_lo, s.ci_hi, color=c, lw=2, alpha=.45, zorder=2)
    ax.scatter(s.auc, s.y, s=70, color=c, zorder=3,
               edgecolor="#fcfcfb", linewidth=1.5, label=g)
    for _, r in s.iterrows():
        ax.text(r.ci_hi + .012, r.y, f"{r.auc:.3f}", va="center",
                fontsize=9, color=INK)

ax.set_yticks(d.y); ax.set_yticklabels(d.label, fontsize=9.5)
ax.set_xlim(0.35, 0.80); ax.set_xlabel("AUC  (95% CI)", color="#52514e")
ax.set_ylim(-0.8, d.y.max() + 1.6)
ax.xaxis.grid(True, color=GRID, lw=.8); ax.set_axisbelow(True)
for sp in ("top","right","left"): ax.spines[sp].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.tick_params(colors=MUTED, length=0)
ax.set_title("Transfer survives within a country, not across countries",
             fontsize=12.5, color=INK, pad=14, loc="left")
ax.legend(frameon=False, loc="lower right", fontsize=9.5)

plt.tight_layout()
plt.savefig(f"{ROOT}/outputs/figures/rq2_transfer.png", dpi=200,
            bbox_inches="tight", facecolor="#fcfcfb")
plt.show()

In [ ]:
d = d.sort_values(["cross_country","auc"], ascending=[False, True]).reset_index(drop=True)
d["grp"] = np.where(d.cross_country, "Cross-country", "Within-country")

ypos, y = [], 0.0
prev = d.grp.iloc[0]
for g in d.grp:
    if g != prev: y += 1.6; prev = g
    ypos.append(y); y += 1.0
d["y"] = ypos

fig, ax = plt.subplots(figsize=(7.6, 5.6), dpi=160)
ax.set_facecolor("#fcfcfb"); fig.patch.set_facecolor("#fcfcfb")
ax.axvline(0.5, color=MUTED, ls="--", lw=1, zorder=1)
ax.text(0.5, d.y.max()+1.3, "chance", color=MUTED, fontsize=9,
        ha="center", va="bottom")

for g, c in [("Within-country", BLUE), ("Cross-country", ORANGE)]:
    s = d[d.grp == g]
    ax.hlines(s.y, s.ci_lo, s.ci_hi, color=c, lw=2, alpha=.45, zorder=2)
    ax.scatter(s.auc, s.y, s=70, color=c, zorder=3,
               edgecolor="#fcfcfb", linewidth=1.5)
    for _, r in s.iterrows():
        ax.text(r.ci_hi + .012, r.y, f"{r.auc:.3f}", va="center",
                fontsize=9, color=INK)
    # direct group label above the block, in the group's colour
    ax.text(0.355, s.y.max() + 0.55, g.upper(), color=c, fontsize=9.5,
            fontweight="bold", ha="left", va="center")

ax.set_yticks(d.y); ax.set_yticklabels(d.label, fontsize=9.5)
ax.set_xlim(0.35, 0.80); ax.set_xlabel("AUC  (95% CI)", color="#52514e")
ax.set_ylim(-0.8, d.y.max() + 1.7)
ax.xaxis.grid(True, color=GRID, lw=.8); ax.set_axisbelow(True)
for sp in ("top","right","left"): ax.spines[sp].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.tick_params(colors=MUTED, length=0)
ax.set_title("Transfer survives within a country, not across countries",
             fontsize=12.5, color=INK, pad=16, loc="left")

plt.tight_layout()
plt.savefig(f"{ROOT}/outputs/figures/rq2_transfer.png", dpi=200,
            bbox_inches="tight", facecolor="#fcfcfb")
plt.show()

In [ ]:
def risk_coverage(y, p, grid=np.linspace(0.05, 1.0, 20)):
    """Accuracy among the most-confident fraction, as coverage varies."""
    y, p = np.asarray(y), np.asarray(p)
    conf = np.abs(p - 0.5)                 # distance from the decision boundary
    order = np.argsort(-conf)              # most confident first
    out = []
    for cov in grid:
        k = max(5, int(round(cov * len(y))))
        idx = order[:k]
        acc = ((p[idx] >= .5).astype(int) == y[idx]).mean()
        rec = (((p[idx] >= .5).astype(int) == 1) & (y[idx] == 1)).sum() / max((y[idx]==1).sum(),1)
        out.append(dict(coverage=round(k/len(y),3), n_kept=k,
                        referral_rate=round(1-k/len(y),3),
                        acc=round(acc,3), rec_hr=round(rec,3)))
    return pd.DataFrame(out)

tr = subsample_patients(man_all[man_all.site=="LKA_R"], N_TRAIN)
curves = {}
for te_site in ["LKA_N", "MYS_sample"]:
    te = man_all[man_all.site==te_site].reset_index(drop=True)
    p = cnn(tr, te)
    c = risk_coverage(te.risk_label.values, p)
    c["test"] = te_site
    curves[te_site] = c
    print(f"\n{te_site}: accuracy at selected referral rates")
    print(c[c.coverage.isin(c.coverage.iloc[[19,14,9,4]])][
        ["referral_rate","n_kept","acc","rec_hr"]].to_string(index=False))

rc = pd.concat(curves.values(), ignore_index=True)
rc.to_csv(f"{ROOT}/outputs/tables/rq4_risk_coverage.csv", index=False)

In [ ]:
rc = pd.read_csv(f"{ROOT}/outputs/tables/rq4_risk_coverage.csv")

fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.2), dpi=160, sharey=True)
fig.patch.set_facecolor("#fcfcfb")

for ax, (site, c, title) in zip(axes, [
        ("LKA_N", BLUE,   "Within-country  (LKA_R \u2192 LKA_N)"),
        ("MYS_sample", ORANGE, "Cross-country  (LKA_R \u2192 MYS)")]):
    s = rc[rc.test == site].sort_values("referral_rate")
    ax.set_facecolor("#fcfcfb")
    ax.plot(s.referral_rate, s.acc, color=c, lw=2.5, label="Accuracy")
    ax.plot(s.referral_rate, s.rec_hr, color=c, lw=2, ls=":",
            label="Recall, high-risk")
    ax.axhline(0.5, color=MUTED, ls="--", lw=1)
    ax.set_title(title, fontsize=11, color=INK, loc="left", pad=10)
    ax.set_xlabel("Referral rate", color="#52514e")
    ax.set_xlim(0, .8); ax.set_ylim(0, 1.0)
    ax.grid(True, color=GRID, lw=.8); ax.set_axisbelow(True)
    for sp in ("top","right"): ax.spines[sp].set_visible(False)
    ax.spines["left"].set_color("#c3c2b7"); ax.spines["bottom"].set_color("#c3c2b7")
    ax.tick_params(colors=MUTED, length=0)
    ax.legend(frameon=False, fontsize=9, loc="lower left")

axes[0].set_ylabel("Performance on retained cases", color="#52514e")
fig.suptitle("Abstention works within a country and inverts across countries",
             fontsize=12.5, color=INK, x=0.012, ha="left", y=1.02)
plt.tight_layout()
plt.savefig(f"{ROOT}/outputs/figures/rq4_risk_coverage.png", dpi=200,
            bbox_inches="tight", facecolor="#fcfcfb")
plt.show()

In [ ]:
!cd /content && zip -qr img512.zip img512
!cp /content/img512.zip "{ROOT}/data/"
!ls -lh "{ROOT}/data/img512.zip"

In [ ]:
tr = subsample_patients(man_all[man_all.site=="LKA_R"], N_TRAIN)

preds = {}
for te_site in ["LKA_N", "MYS_sample"]:
    te = man_all[man_all.site==te_site].reset_index(drop=True)
    te = te.copy()
    te["prob"] = cnn(tr, te)
    te["pred"] = (te.prob >= .5).astype(int)
    te["correct"] = (te.pred == te.risk_label).astype(int)
    preds[te_site] = te

allp = pd.concat(preds.values(), ignore_index=True)
allp.to_csv(f"{ROOT}/outputs/tables/predictions_trainR.csv", index=False)

for site, te in preds.items():
    print(f"\n{'='*62}\n{site}  (n={len(te)}, acc={te.correct.mean():.3f})\n{'='*62}")
    g = (te.groupby("clinical_dx")
           .agg(n=("correct","size"),
                truth_high_risk=("risk_label","mean"),
                mean_prob=("prob","mean"),
                accuracy=("correct","mean"))
           .round(3)
           .sort_values("n", ascending=False))
    print(g[g.n >= 5].to_string())

In [ ]:
# 1. where does the model put its probability mass, by lesion type?
print("\n\n=== mean predicted risk by diagnosis class, per site ===")
piv = allp.pivot_table(index="diagnosis", columns="site",
                       values="prob", aggfunc="mean").round(3)
cnt = allp.pivot_table(index="diagnosis", columns="site",
                       values="prob", aggfunc="size")
print(piv.to_string())
print("\ncounts:")
print(cnt.to_string())

# 2. the specific failures — high-risk cases called benign, most confidently
print("\n\n=== missed high-risk cases (model most confident they were safe) ===")
for site, te in preds.items():
    miss = te[(te.risk_label==1) & (te.pred==0)].nsmallest(8, "prob")
    print(f"\n{site}  — {len(te[(te.risk_label==1)&(te.pred==0)])} missed of "
          f"{int(te.risk_label.sum())} high-risk")
    print(miss[["filename","diagnosis","clinical_dx","prob"]].to_string(index=False))

In [ ]:
def norm_dx(s):
    s = str(s).upper().replace(" ", "")
    if "OSF" in s: return "OSF"
    if "OLP" in s or "LICHEN" in s: return "OLP"
    if "CANCER" in s or s == "OCA": return "Cancer"
    if "NORMAL" in s or "NOLESION" in s: return "Normal"
    if "LEUK" in s: return "Leukoplakia"
    return "Other"

allp["dx_group"] = allp.clinical_dx.map(norm_dx)

print("=== AUC restricted to matched lesion groups ===")
rows = []
for site in ["LKA_N", "MYS_sample"]:
    s = allp[allp.site == site]
    for grp in ["OLP", "Normal", "OSF", "Cancer"]:
        sub = s[s.dx_group.isin([grp, "Normal"])] if grp != "Normal" else None
        if sub is None or sub.risk_label.nunique() < 2 or len(sub) < 25: continue
        a = roc_auc_score(sub.risk_label, sub.prob)
        lo, hi = boot_auc(sub.risk_label.values, sub.prob.values)
        rows.append(dict(site=site, task=f"{grp} vs Normal", n=len(sub),
                         auc=round(a,3), ci_lo=lo, ci_hi=hi))
        print(f"{site:12s} {grp:12s} vs Normal  n={len(sub):4d}  "
              f"auc={a:.3f} [{lo}, {hi}]")

pd.DataFrame(rows).to_csv(f"{ROOT}/outputs/tables/rq2_by_lesion.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

te = preds["LKA_N"]
missed_olp = te[(te.clinical_dx.str.contains("OLP")) & (te.pred==0)].nsmallest(4, "prob")
caught_osf = te[(te.clinical_dx=="OSF") & (te.pred==1)].nlargest(4, "prob")

fig, axes = plt.subplots(2, 4, figsize=(13, 7), dpi=130)
fig.patch.set_facecolor("#fcfcfb")

for ax, (_, r) in zip(axes[0], missed_olp.iterrows()):
    ax.imshow(Image.open(r.filepath))
    ax.set_title(f"{r.clinical_dx}\npredicted risk {r.prob:.2f}  MISSED",
                 fontsize=9.5, color="#eb6834")
    ax.axis("off")

for ax, (_, r) in zip(axes[1], caught_osf.iterrows()):
    ax.imshow(Image.open(r.filepath))
    ax.set_title(f"{r.clinical_dx}\npredicted risk {r.prob:.2f}  caught",
                 fontsize=9.5, color="#2a78d6")
    ax.axis("off")

fig.suptitle("Top: lichen planus the model calls benign   |   "
             "Bottom: submucous fibrosis it flags correctly",
             fontsize=12, color="#0b0b0b", y=0.98)
plt.tight_layout()
plt.savefig(f"{ROOT}/outputs/figures/error_analysis.png", dpi=180,
            bbox_inches="tight", facecolor="#fcfcfb")
plt.show()

In [ ]:
te = preds["LKA_N"]
te["betel_yes"] = te.betel.astype(str).str.strip().str.lower().eq("yes")

print("=== predicted risk by betel status, WITHIN each diagnosis ===")
g = (te.groupby(["diagnosis","betel_yes"])
       .agg(n=("prob","size"), mean_prob=("prob","mean"),
            pred_high_rate=("pred","mean"))
       .round(3))
print(g.to_string())

print("\n=== the decisive test: healthy mucosa only ===")
h = te[te.risk_label == 0]
print(h.groupby("betel_yes").agg(n=("prob","size"),
                                 mean_prob=("prob","mean"),
                                 false_pos_rate=("pred","mean")).round(3))

from scipy.stats import mannwhitneyu
a = h[h.betel_yes].prob; b = h[~h.betel_yes].prob
if len(a) > 5 and len(b) > 5:
    u, p = mannwhitneyu(a, b)
    print(f"\nMann-Whitney U on benign cases: p = {p:.2e}")
    print(f"  betel chewers    mean predicted risk {a.mean():.3f}  (n={len(a)})")
    print(f"  non-chewers      mean predicted risk {b.mean():.3f}  (n={len(b)})")

In [ ]:
import numpy as np
from PIL import Image

def mask_lower_third(src, dst):
    """Crude occlusion: blank the region where teeth usually sit."""
    if os.path.exists(dst): return
    im = Image.open(src).convert("RGB")
    a = np.asarray(im).copy()
    h = a.shape[0]
    a[int(h*0.55):, :, :] = 128          # grey out lower 45%
    Image.fromarray(a).save(dst, "JPEG", quality=92)

MASK = "/content/img512_masked"
os.makedirs(MASK, exist_ok=True)
sub = man_all[man_all.site.isin(["LKA_R","LKA_N"])].copy()
sub["masked"] = [f"{MASK}/{os.path.basename(p)}" for p in sub.filepath]
for s, d in tqdm(list(zip(sub.filepath, sub.masked)), desc="mask"):
    mask_lower_third(s, d)

# retrain on masked R, test on masked N
trm = sub[sub.site=="LKA_R"].copy(); trm["filepath"] = trm.masked
trm = subsample_patients(trm, N_TRAIN)
tem = sub[sub.site=="LKA_N"].copy(); tem["filepath"] = tem.masked
p_masked = cnn(trm, tem)
print("masked  LKA_R->LKA_N  AUC =", round(roc_auc_score(tem.risk_label, p_masked), 3))
print("unmasked (earlier)                 = 0.689")

In [ ]:
def mask_upper(src, dst, frac=0.45):
    if os.path.exists(dst): return
    a = np.asarray(Image.open(src).convert("RGB")).copy()
    a[:int(a.shape[0]*frac), :, :] = 128
    Image.fromarray(a).save(dst, "JPEG", quality=92)

MASKU = "/content/img512_maskedup"
os.makedirs(MASKU, exist_ok=True)
sub["maskedup"] = [f"{MASKU}/{os.path.basename(p)}" for p in sub.filepath]
for s, d in tqdm(list(zip(sub.filepath, sub.maskedup)), desc="mask-up"):
    mask_upper(s, d)

tru = sub[sub.site=="LKA_R"].copy(); tru["filepath"] = tru.maskedup
tru = subsample_patients(tru, N_TRAIN)
teu = sub[sub.site=="LKA_N"].copy(); teu["filepath"] = teu.maskedup
p_up = cnn(tru, teu)

print(f"unmasked        0.689")
print(f"lower masked    0.637")
print(f"upper masked    {roc_auc_score(teu.risk_label, p_up):.3f}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
ROOT = "/content/drive/MyDrive/oral_cancer_crosssite"
sys.path.append(f"{ROOT}/src")
!pip install -q timm albumentations
from pipeline import *
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score

# restore the 512px image cache from Drive
!unzip -q "{ROOT}/data/img512.zip" -d /content/
man_all = pd.read_csv(f"{ROOT}/outputs/tables/manifest_all.csv")
N_TRAIN = 640
cache = build_feature_cache(man_all.filepath.tolist())
cnn = make_cnn_predictor(arch="resnet18", epochs=8, bs=32, lr=3e-4)
print("ready:", len(man_all), "images")

In [ ]:
def subsample_patients(df, n_img, seed=0):
    if len(df) <= n_img: return df
    rng = np.random.default_rng(seed)
    keep, tot = [], 0
    for p in rng.permutation(df.patient_id.unique()):
        k = (df.patient_id == p).sum()
        if tot + k > n_img: break
        keep.append(p); tot += k
    return df[df.patient_id.isin(keep)].reset_index(drop=True)

In [ ]:
# ============================================================
# COMPLETE THE PROJECT — RQ3, RQ2b, and the statistics
# ~25 min total. If Colab restarted, re-run your setup cell first.
# ============================================================
from sklearn.model_selection import GroupShuffleSplit
from scipy.stats import mannwhitneyu
from itertools import combinations

# ---------- PART 1: RQ3 few-shot fine-tuning ----------
print("="*60 + "\nRQ3: few-shot adaptation\n" + "="*60)

def split_target(df, test_frac=0.5, seed=0):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_frac, random_state=seed)
    pool_i, test_i = next(gss.split(df, df.risk_label, df.patient_id))
    return df.iloc[pool_i].reset_index(drop=True), df.iloc[test_i].reset_index(drop=True)

def draw_k(pool, k, seed=0):
    if k == 0: return pool.iloc[0:0]
    rng = np.random.default_rng(seed)
    keep, tot = [], 0
    for pid in rng.permutation(pool.patient_id.unique()):
        n = (pool.patient_id == pid).sum()
        if tot + n > k and keep: break
        keep.append(pid); tot += n
        if tot >= k: break
    return pool[pool.patient_id.isin(keep)]

SOURCE = subsample_patients(man_all[man_all.site == "LKA_R"], N_TRAIN)
rq3 = []
for target, ks in [("LKA_S", [0, 10, 25, 50, 100]),
                   ("MYS_sample", [0, 10, 25, 50])]:
    tgt = man_all[man_all.site == target].reset_index(drop=True)
    for seed in [0, 1]:
        pool, test = split_target(tgt, 0.5, seed)
        for k in ks:
            add = draw_k(pool, k, seed)
            tr = pd.concat([SOURCE, add], ignore_index=True) if k else SOURCE
            auc = roc_auc_score(test.risk_label, cnn(tr, test))
            rq3.append(dict(target=target, seed=seed, k=k, k_actual=len(add),
                            n_test=len(test), auc=round(auc, 3)))
            print(f"  {target:12s} seed{seed} k={k:3d} (got {len(add):3d})  auc={auc:.3f}")

rq3 = pd.DataFrame(rq3)
rq3.to_csv(f"{ROOT}/outputs/tables/rq3_fewshot_raw.csv", index=False)
curve = rq3.groupby(["target","k"]).auc.agg(["mean","std"]).round(3)
curve.to_csv(f"{ROOT}/outputs/tables/rq3_fewshot_curve.csv")
print("\nLEARNING CURVE (mean over 2 seeds):")
print(curve.to_string())

# ---------- PART 2: RQ2b what explains the gap ----------
print("\n" + "="*60 + "\nRQ2b: explanatory factors\n" + "="*60)

def norm_dx(s):
    s = str(s).upper().replace(" ", "")
    if "OSF" in s: return "OSF"
    if "OLP" in s or "LICHEN" in s: return "OLP"
    if "CANCER" in s or s == "OCA": return "Cancer"
    if "NORMAL" in s or "NOLESION" in s: return "Normal"
    return "Other"

m = man_all.copy(); m["dx_group"] = m.clinical_dx.map(norm_dx)
prof = (m.groupby("site")
          .agg(n=("filename","size"), high_risk=("risk_label","mean"),
               olp_share=("dx_group", lambda s:(s=="OLP").mean()),
               osf_share=("dx_group", lambda s:(s=="OSF").mean()))
          .join(pd.read_csv(f"{ROOT}/outputs/tables/site_profile.csv")
                  .set_index("site")[["med_w","brightness"]]))
print("SITE PROFILE:"); print(prof.round(3).to_string())

mat = pd.read_csv(f"{ROOT}/outputs/tables/rq2_matrix_fixed_budget.csv")
for v in ["brightness","med_w","high_risk","olp_share","osf_share"]:
    mat[f"d_{v}"] = [abs(prof.loc[a,v]-prof.loc[b,v]) for a,b in zip(mat.train, mat.test)]

from scipy.stats import pearsonr
rows = [dict(variable=v, pearson_r=round(pearsonr(mat[v], mat.auc)[0],3),
             p=round(pearsonr(mat[v], mat.auc)[1],4))
        for v in ["d_brightness","d_med_w","d_high_risk","d_olp_share","d_osf_share"]]
expl = pd.DataFrame(rows).sort_values("pearson_r")
expl.to_csv(f"{ROOT}/outputs/tables/rq2b_explanatory.csv", index=False)
print("\nCORRELATION WITH TRANSFER AUC (n=12 pairs):")
print(expl.to_string(index=False))

# ---------- PART 3: statistics ----------
print("\n" + "="*60 + "\nSTATISTICS\n" + "="*60)
w = mat[~mat.cross_country].auc.values; c = mat[mat.cross_country].auc.values
u, p_mw = mannwhitneyu(w, c, alternative="greater")
print(f"within  mean={w.mean():.3f}  {sorted(w)}")
print(f"cross   mean={c.mean():.3f}  {sorted(c)}")
print(f"Mann-Whitney U={u}, one-sided p={p_mw:.4f}")

allv = np.concatenate([w,c]); obs = w.mean()-c.mean()
diffs = np.array([allv[list(i)].mean()-np.delete(allv,list(i)).mean()
                  for i in combinations(range(12),6)])
print(f"exact permutation: diff={obs:.3f}, p={(diffs>=obs).mean():.4f} "
      f"({len(diffs)} splits)")

print("\nSEED VARIANCE (3 runs):")
sv = []
for tr_s, te_s in [("LKA_R","LKA_N"), ("LKA_R","MYS_sample")]:
    tr = subsample_patients(man_all[man_all.site==tr_s], N_TRAIN)
    te = man_all[man_all.site==te_s].reset_index(drop=True)
    a = [roc_auc_score(te.risk_label,
                       make_cnn_predictor(epochs=8,bs=32,lr=3e-4,seed=s)(tr,te))
         for s in [0,1,2]]
    sv.append(dict(direction=f"{tr_s}->{te_s}", mean=round(np.mean(a),3),
                   sd=round(np.std(a),3), runs=[round(x,3) for x in a]))
    print(" ", sv[-1])
pd.DataFrame(sv).to_csv(f"{ROOT}/outputs/tables/seed_variance.csv", index=False)
print("\nDONE — paste all output back.")

In [ ]:
import sys; sys.path.insert(0, '/content/drive/MyDrive/oral_cancer_crosssite/src')
import finish_project as F

F.step0_load()                                # manifest + leakage sanity, seconds
F.step1_rq3_leakage_check(adapt_df, eval_df)  # RUN FIRST — seconds
F.step2_rerun_rq1()                           # retrains CNN, slow
F.step3_rerun_rq2()                           # retrains CNN x12, slowest
F.step4_permutation()
F.step5_budget_direction_check()
F.step6_figure()
F.step7_audit()

In [ ]:
import os
ROOT = '/content/drive/MyDrive/oral_cancer_crosssite'
print('drive mounted:', os.path.exists('/content/drive/MyDrive'))
print('ROOT exists  :', os.path.exists(ROOT))
print('src exists   :', os.path.exists(f'{ROOT}/src'))
print('src contents :', os.listdir(f'{ROOT}/src') if os.path.exists(f'{ROOT}/src') else '-')

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

In [ ]:
import os
ROOT = '/content/drive/MyDrive/oral_cancer_crosssite'
print('drive mounted:', os.path.exists('/content/drive/MyDrive'))
print('ROOT exists  :', os.path.exists(ROOT))
print('src exists   :', os.path.exists(f'{ROOT}/src'))
print('src contents :', os.listdir(f'{ROOT}/src') if os.path.exists(f'{ROOT}/src') else '-')

In [ ]:
import sys, importlib
if f'{ROOT}/src' not in sys.path:
    sys.path.insert(0, f'{ROOT}/src')
import finish_project as F
importlib.reload(F)
F.step0_load()

In [ ]:
import sys, importlib
importlib.invalidate_caches()          # this is the fix
if f'{ROOT}/src' not in sys.path:
    sys.path.insert(0, f'{ROOT}/src')
import finish_project as F
F.step0_load()

In [ ]:
import sys, importlib
importlib.invalidate_caches()          # this is the fix
if f'{ROOT}/src' not in sys.path:
    sys.path.insert(0, f'{ROOT}/src')
import finish_project as F
F.step0_load()

In [ ]:
import sys, os, shutil, importlib, importlib.util

path = f'{ROOT}/src/finish_project.py'
src  = open(path).read()
print('file on disk is the NEW version:', 'SITES_EXT' in src)
print('file size:', len(src), 'bytes')

# drop the in-memory copy and any compiled cache
sys.modules.pop('finish_project', None)
shutil.rmtree(f'{ROOT}/src/__pycache__', ignore_errors=True)
importlib.invalidate_caches()

# load straight from the file, bypassing the import system
spec = importlib.util.spec_from_file_location('finish_project', path)
F = importlib.util.module_from_spec(spec)
spec.loader.exec_module(F)

print('SITES    :', F.SITES)
print('SITES_EXT:', F.SITES_EXT)

In [ ]:
F.step0_load()

In [ ]:
import os, zipfile, torch

# 1. restore the resized image cache from your Drive backup
need = not os.path.exists('/content/img512') or len(os.listdir('/content/img512')) < 3000
if need:
    with zipfile.ZipFile(f'{ROOT}/data/img512.zip') as z:
        z.extractall('/content/')
print('images present:', len(os.listdir('/content/img512')))

# 2. confirm the manifest's paths actually resolve
import pandas as pd
df = pd.read_csv(f'{TAB}/manifest_all.csv')
missing = (~df.path512.map(os.path.exists)).sum()
print('manifest paths missing on disk:', missing)
assert missing == 0, 'image cache does not match the manifest'

# 3. GPU check
print('CUDA available:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
import os, pandas as pd, torch

df = pd.read_csv(f'{F.TAB}/manifest_all.csv')
missing = (~df.path512.map(os.path.exists)).sum()
print('manifest paths missing on disk:', missing)
assert missing == 0, 'image cache does not match the manifest'

print('CUDA available:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
rq2 = F.step3_rerun_rq2()

In [ ]:
import inspect
from pipeline import make_cnn_predictor

print('signature:', inspect.signature(make_cnn_predictor))

def _cnn_safe():
    want = dict(epochs=8, bs=32, lr=3e-4, wd=1e-4,
                weight_decay=1e-4, batch_size=32, n_epochs=8)
    ok = set(inspect.signature(make_cnn_predictor).parameters)
    kw = {k: v for k, v in want.items() if k in ok}
    print('calling make_cnn_predictor with', kw)
    return make_cnn_predictor(**kw)

F._cnn = _cnn_safe
rq2 = F.step3_rerun_rq2()

In [ ]:
F.step4_permutation()
F.step5_budget_direction_check()
F.step6_figure()

In [ ]:
import pandas as pd, numpy as np
rq2 = pd.read_csv(f'{F.TAB}/rq2_matrix_patientboot.csv')
rq2['direction'] = np.where(~rq2.cross, 'within-country',
                   np.where(rq2.train.str.startswith('MYS'),
                            'MYS to LKA  (train 100)',
                            'LKA to MYS  (train 389-639)'))
print(rq2.groupby('direction')
         .agg(n=('auc','size'), mean_auc=('auc','mean'),
              min_auc=('auc','min'), max_auc=('auc','max')).round(3).to_string())
rq2.to_csv(f'{F.TAB}/rq2_matrix_patientboot.csv', index=False)

In [ ]:
rq2_5 = F.step3_rerun_rq2(F.SITES_EXT)
F.step4_permutation(f'{F.TAB}/rq2_matrix_patientboot_5site.csv')

In [ ]:
F.step2_rerun_rq1()   # within-site CV with patient-level intervals
F.step7_audit()       # confirms every promised metric is in the tables

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

def rq1_pooled(df, site, fit_predict, n_splits=5, seed=0):
    g = df[df.site == site].reset_index(drop=True)
    y, groups = g.risk_label.values, g.patient_id.values
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    Y, P, G = [], [], []
    for tr, te in sgkf.split(g, y, groups):
        assert not (set(groups[tr]) & set(groups[te])), 'patient leaked'
        P.append(np.asarray(fit_predict(g.iloc[tr], g.iloc[te])))
        Y.append(y[te]); G.append(groups[te])
    Y, P, G = np.concatenate(Y), np.concatenate(P), np.concatenate(G)
    out = F.score(Y, P, groups=G); out['site'] = site
    return out, pd.DataFrame({'site': site, 'patient_id': G, 'y': Y, 'p': P})

cnn, rows, preds = F._cnn(), [], []
for s in F.SITES:
    o, pr = rq1_pooled(F._df_or_die(), s, cnn)
    rows.append(o); preds.append(pr)

rq1 = pd.DataFrame(rows)[['site','auc','lo','hi','accuracy','macro_f1','n','n_patients']]
rq1.to_csv(f'{F.TAB}/rq1_within_site_pooled.csv', index=False)
pd.concat(preds).to_csv(f'{F.TAB}/rq1_oof_predictions.csv', index=False)
print(rq1.round(3).to_string(index=False))

In [ ]:
a, e = set(adapt_df.patient_id), set(eval_df.patient_id)
print('overlap:', len(a & e), '->', 'OK' if not (a & e) else 'LEAK')

In [ ]:
import pandas as pd
r3 = pd.read_csv(f'{F.TAB}/rq3_fewshot_raw.csv')
print(r3.columns.tolist())
print(r3.head(8).to_string(index=False))

In [ ]:
import importlib, importlib.util, os
ROOT = '/content/drive/MyDrive/oral_cancer_crosssite'
print('file there:', os.path.exists(f'{ROOT}/src/finish_rq34.py'))

importlib.invalidate_caches()
spec = importlib.util.spec_from_file_location('finish_rq34', f'{ROOT}/src/finish_rq34.py')
R = importlib.util.module_from_spec(spec); spec.loader.exec_module(R)

cnn = F._cnn()
F.step0_load()
R.bind(F, cnn)

In [ ]:
from google.colab import files
import shutil, os

ROOT = '/content/drive/MyDrive/oral_cancer_crosssite'
up = files.upload()                      # pick finish_rq34.py from Downloads
name = list(up.keys())[0]
shutil.move(name, f'{ROOT}/src/finish_rq34.py')
print('in place:', os.path.exists(f'{ROOT}/src/finish_rq34.py'))
print('src:', sorted(os.listdir(f'{ROOT}/src')))

In [ ]:
import os, glob, shutil
from google.colab import drive

drive.mount('/content/drive')                     # approve if prompted
ROOT = '/content/drive/MyDrive/oral_cancer_crosssite'
print('ROOT exists:', os.path.exists(ROOT))
print('src  exists:', os.path.exists(f'{ROOT}/src'))

# find the uploaded copy wherever Colab put it
hits = [p for d in ('.', '/content', '/root', os.getcwd())
        for p in glob.glob(os.path.join(d, 'finish_rq34*.py'))]
print('found:', hits)

if hits:
    os.makedirs(f'{ROOT}/src', exist_ok=True)
    shutil.copy(hits[0], f'{ROOT}/src/finish_rq34.py')
    print('copied from', hits[0])

print('in place:', os.path.exists(f'{ROOT}/src/finish_rq34.py'))
print('src:', sorted(os.listdir(f'{ROOT}/src')) if os.path.exists(f'{ROOT}/src') else '-')

In [ ]:
import importlib, importlib.util
importlib.invalidate_caches()
spec = importlib.util.spec_from_file_location('finish_rq34', f'{ROOT}/src/finish_rq34.py')
R = importlib.util.module_from_spec(spec); spec.loader.exec_module(R)

In [ ]:
import os, sys, zipfile, inspect, importlib, importlib.util
import pandas as pd, numpy as np, torch

ROOT = '/content/drive/MyDrive/oral_cancer_crosssite'
TAB, FIG = f'{ROOT}/outputs/tables', f'{ROOT}/outputs/figures'
if f'{ROOT}/src' not in sys.path:
    sys.path.insert(0, f'{ROOT}/src')

# 1. restore the resized image cache (wiped by the remount)
if not os.path.exists('/content/img512') or len(os.listdir('/content/img512')) < 3000:
    with zipfile.ZipFile(f'{ROOT}/data/img512.zip') as z:
        z.extractall('/content/')
print('images:', len(os.listdir('/content/img512')))

# 2. load both modules straight from file
importlib.invalidate_caches()
def load(name):
    spec = importlib.util.spec_from_file_location(name, f'{ROOT}/src/{name}.py')
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m
F = load('finish_project')
R = load('finish_rq34')

# 3. CNN factory matching your pipeline's real signature (it has no `wd`)
from pipeline import make_cnn_predictor
def _cnn_safe():
    want = dict(epochs=8, bs=32, lr=3e-4)
    ok   = set(inspect.signature(make_cnn_predictor).parameters)
    return make_cnn_predictor(**{k: v for k, v in want.items() if k in ok})
F._cnn = _cnn_safe

# 4. data and GPU
F.step0_load()
print('CUDA:', torch.cuda.is_available())

# 5. bind
cnn = F._cnn()
R.bind(F, cnn)

In [ ]:
rq4 = R.run_rq4()

In [ ]:
rq3, curve = R.run_rq3(source='LKA_R', targets=['LKA_S', 'MYS_sample'])

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
rq4 = pd.read_csv(f'{F.TAB}/rq4_risk_coverage_patientboot.csv')
BLUE, ORANGE, GREY = '#2a78d6', '#eb6834', '#898781'

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0), dpi=200, sharey=True)
for ax, lbl in zip(axes, ['within-country', 'cross-country']):
    d = rq4[rq4.comparison == lbl].sort_values('referral_rate')
    ax.plot(d.referral_rate, d.accuracy, 'o-', color=BLUE, ms=4, label='accuracy (retained)')
    ax.plot(d.referral_rate, d.high_risk_recall, 's-', color=ORANGE, ms=4, label='high-risk recall')
    ax.axhline(0.5, color=GREY, ls='--', lw=1)
    ax.set_title(f'{lbl} — {d.test.iloc[0]}', fontsize=10)
    ax.set_xlabel('referral rate')
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
axes[0].set_ylabel('score on retained cases')
axes[0].legend(frameon=False, fontsize=8, loc='upper left')
plt.tight_layout()
plt.savefig(f'{F.FIG}/rq4_riskcoverage_patientboot.png', bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
df = F._df_or_die()

print('clinical_dx by site, high-risk cases only:')
hr = df[df.risk_label == 1]
for s in ['LKA_R', 'LKA_N', 'LKA_S', 'MYS_sample']:
    vc = hr[hr.site == s].clinical_dx.value_counts()
    print(f'\n--- {s}  (n={len(hr[hr.site==s])}) ---')
    print(vc.head(10).to_string())

In [ ]:
import numpy as np, pandas as pd

def attach(test_site, src='LKA_R'):
    pr = pd.read_csv(f'{F.TAB}/predictions_{src}_to_{test_site}.csv')
    te = df[df.site == test_site].reset_index(drop=True)
    assert len(pr) == len(te), 'length mismatch'
    assert (pr.patient_id.values == te.patient_id.values).all(), 'row order mismatch'
    out = te.copy(); out['p'] = pr.p.values
    assert (pr.y.values == out.risk_label.values).all()
    return out

def lesion_auc(d, pattern, label):
    """High-risk cases matching `pattern`, scored against all low-risk cases."""
    pos = (d.risk_label == 1) & d.clinical_dx.str.contains(pattern, case=False, na=False)
    sub = d[pos | (d.risk_label == 0)]
    if pos.sum() < 5 or (sub.risk_label == 0).sum() < 5:
        print(f'{label:34s} too few cases (pos={int(pos.sum())})'); return
    r = F.score(sub.risk_label.values, sub.p.values, groups=sub.patient_id.values)
    print(f'{label:34s} AUC {r["auc"]:.3f} [{r["lo"]:.3f}, {r["hi"]:.3f}]  '
          f'pos={int(pos.sum())}  neg={int((sub.risk_label==0).sum())}')

for site in ['MYS_sample', 'LKA_N']:
    d = attach(site)
    print(f'\n=== LKA_R -> {site} ===')
    lesion_auc(d, 'lichen|olp',            'oral lichen planus')
    lesion_auc(d, 'submucous|fibrosis|osf','submucous fibrosis')
    lesion_auc(d, 'leukoplakia',           'leukoplakia')

In [ ]:
d = attach('LKA_N')
d['correct'] = ((d.p >= 0.5).astype(int) == d.risk_label).astype(int)

acc = (d.groupby('clinical_dx')
         .agg(n=('correct', 'size'), accuracy=('correct', 'mean'))
         .query('n >= 10')
         .sort_values('accuracy'))
print('per-lesion accuracy, LKA_R -> LKA_N:')
print(acc.round(3).to_string())

print()
lesion_auc(d, 'oral cancer', 'oral cancer')

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import chi2_contingency
from sklearn.metrics import roc_auc_score
from PIL import Image

df = F._df_or_die()

# --- floors (instant) ---
d = attach('LKA_N')
y = d.risk_label.values
print('majority-class accuracy:', round(max(y.mean(), 1 - y.mean()), 3))
rng = np.random.default_rng(0)
print('random-predictor AUC   :', round(roc_auc_score(y, rng.random(len(y))), 3))

# --- betel association (instant) ---
ct = pd.crosstab(df.betel, df.risk_label)
chi2, p, _, _ = chi2_contingency(ct)
print('\nbetel x risk_label:'); print(ct.to_string())
print(f'chi2 = {chi2:.2f}, p = {p:.2e}')

# --- site-level factors (~2 min: reads every image once) ---
bright = {}
for site, g in df.groupby('site'):
    vals = [np.asarray(Image.open(p).convert('L')).mean() for p in g.path512]
    bright[site] = float(np.mean(vals))
prof = df.groupby('site').apply(lambda g: pd.Series({
    'brightness': bright[g.name],
    'olp_share': g.clinical_dx.str.contains('olp|lichen', case=False, na=False).mean(),
    'osf_share': g.clinical_dx.str.contains('osf|submucous', case=False, na=False).mean(),
    'n_img': len(g)}), include_groups=False)
print('\nsite profile:'); print(prof.round(3).to_string())

rq2 = pd.read_csv(f'{F.TAB}/rq2_matrix_patientboot.csv')
for col in ['brightness', 'olp_share', 'osf_share', 'n_img']:
    rq2['d_' + col] = [abs(prof.loc[a, col] - prof.loc[b, col])
                       for a, b in zip(rq2.train, rq2.test)]
print('\ncorrelation of each site difference with transfer AUC:')
print(rq2[['d_brightness','d_olp_share','d_osf_share','d_n_img']]
        .corrwith(rq2.auc).round(3).to_string())

In [ ]:
from scipy.stats import mannwhitneyu
d = attach('LKA_N')
d['flag'] = (d.p >= 0.5).astype(int)

for name, sub in [('benign / healthy (risk_label 0)', d[d.risk_label == 0]),
                  ('high-risk (risk_label 1)',        d[d.risk_label == 1])]:
    g = (sub.groupby('betel')
            .agg(n=('p', 'size'), mean_pred_risk=('p', 'mean'), flagged=('flag', 'mean')))
    print(f'\n{name}:'); print(g.round(3).to_string())
    a = sub.loc[sub.betel == 'Yes', 'p']; b = sub.loc[sub.betel == 'No', 'p']
    if len(a) > 3 and len(b) > 3:
        print('Mann-Whitney U p =', f'{mannwhitneyu(a, b, alternative="greater").pvalue:.2g}')

print('\nnon-chewers only, mean predicted risk by true label:')
print(d[d.betel == 'No'].groupby('risk_label').p.agg(['size', 'mean']).round(3).to_string())

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import (roc_auc_score, precision_recall_fscore_support,
                             accuracy_score)
from scipy.stats import mannwhitneyu, chi2_contingency

# --- floors on LKA_R out-of-fold ---
oof = pd.read_csv(f'{F.TAB}/rq1_oof_predictions.csv')
d0 = oof[oof.site == 'LKA_R']; y = d0.y.values
maj = int(round(y.mean())); yh = np.full_like(y, maj)
_, rc, _, _  = precision_recall_fscore_support(y, yh, average='binary', zero_division=0)
_, _, mf1, _ = precision_recall_fscore_support(y, yh, average='macro',  zero_division=0)
print(f'LKA_R n={len(y)} high-risk={y.mean():.3f}')
print(f'majority -> predicts {maj}, acc {accuracy_score(y, yh):.3f}, AUC 0.500, '
      f'macro-F1 {mf1:.3f}, HR recall {rc:.3f}')
print(f'random   -> AUC {roc_auc_score(y, np.random.default_rng(0).random(len(y))):.3f}')

# --- precision / recall / confusion ---
r1 = pd.read_csv(f'{F.TAB}/rq1_within_site_pooled.csv')
print('\nRQ1:'); print(r1[[c for c in
  ['site','auc','accuracy','precision','recall','macro_f1','tn','fp','fn','tp']
  if c in r1.columns]].round(3).to_string(index=False))

# --- RQ4 missed high-risk counts ---
r4 = pd.read_csv(f'{F.TAB}/rq4_risk_coverage_patientboot.csv')
print('\nRQ4 missed high-risk:')
print(r4[r4.referral_rate.isin([0.0,0.25,0.5,0.75])]
      [['comparison','referral_rate','n_retained','n_high_risk_retained',
        'missed_high_risk','high_risk_recall']].round(3).to_string(index=False))

# --- betel at PATIENT level (their pseudoreplication point) ---
dn = attach('LKA_N')
pat = (dn.groupby('patient_id')
         .agg(betel=('betel','first'), any_hr=('risk_label','max'), mean_p=('p','mean')))
ct = pd.crosstab(pat.betel, pat.any_hr)
chi2, p, _, _ = chi2_contingency(ct)
print(f'\npatient-level betel x any high-risk (n={len(pat)} patients):')
print(ct.to_string()); print(f'chi2={chi2:.2f}, p={p:.3g}')
low = pat[pat.any_hr == 0]
a = low.loc[low.betel=='Yes','mean_p']; b = low.loc[low.betel=='No','mean_p']
print(f'\nlow-risk patients only: chewers n={len(a)} mean_p={a.mean():.3f} | '
      f'non-chewers n={len(b)} mean_p={b.mean():.3f}')
if len(a) > 3:
    print('Mann-Whitney (patient level) p =',
          f'{mannwhitneyu(a, b, alternative="greater").pvalue:.3g}')

In [ ]:
import os, numpy as np, pandas as pd
from google.colab import drive
from sklearn.metrics import (roc_auc_score, precision_recall_fscore_support,
                             accuracy_score)
from scipy.stats import mannwhitneyu, chi2_contingency

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/oral_cancer_crosssite'
TAB  = f'{ROOT}/outputs/tables'
df   = pd.read_csv(f'{TAB}/manifest_all.csv')

def attach(test_site, src='LKA_R'):
    pr = pd.read_csv(f'{TAB}/predictions_{src}_to_{test_site}.csv')
    te = df[df.site == test_site].reset_index(drop=True)
    assert len(pr) == len(te) and (pr.patient_id.values == te.patient_id.values).all()
    out = te.copy(); out['p'] = pr.p.values
    return out

# --- 1 & 2: floors on LKA_R out-of-fold ---
d0 = pd.read_csv(f'{TAB}/rq1_oof_predictions.csv').query("site == 'LKA_R'")
y  = d0.y.values
maj = int(round(y.mean())); yh = np.full_like(y, maj)
_, rc, _, _  = precision_recall_fscore_support(y, yh, average='binary', zero_division=0)
_, _, mf1, _ = precision_recall_fscore_support(y, yh, average='macro',  zero_division=0)
print(f'LKA_R  n={len(y)}  high-risk rate={y.mean():.3f}')
print(f'majority -> predicts class {maj}, accuracy {accuracy_score(y, yh):.3f}, '
      f'AUC 0.500, macro-F1 {mf1:.3f}, high-risk recall {rc:.3f}')
print(f'random   -> AUC {roc_auc_score(y, np.random.default_rng(0).random(len(y))):.3f}')

# --- 3: precision / recall / confusion ---
r1 = pd.read_csv(f'{TAB}/rq1_within_site_pooled.csv')
keep = ['site','auc','accuracy','precision','recall','macro_f1','tn','fp','fn','tp']
print('\nRQ1 within-site:')
print(r1[[c for c in keep if c in r1.columns]].round(3).to_string(index=False))

# --- 4: RQ4 missed high-risk counts ---
r4 = pd.read_csv(f'{TAB}/rq4_risk_coverage_patientboot.csv')
print('\nRQ4 missed high-risk:')
print(r4[r4.referral_rate.isin([0.0, 0.25, 0.5, 0.75])]
      [['comparison','referral_rate','n_retained','n_high_risk_retained',
        'missed_high_risk','high_risk_recall']].round(3).to_string(index=False))

# --- 5: betel at patient level ---
dn  = attach('LKA_N')
pat = (dn.groupby('patient_id')
         .agg(betel=('betel','first'), any_hr=('risk_label','max'), mean_p=('p','mean')))
ct = pd.crosstab(pat.betel, pat.any_hr)
chi2, p, _, _ = chi2_contingency(ct)
print(f'\npatient-level betel x any high-risk  (n={len(pat)} patients):')
print(ct.to_string()); print(f'chi2 = {chi2:.2f}, p = {p:.3g}')

low = pat[pat.any_hr == 0]
a = low.loc[low.betel == 'Yes', 'mean_p']; b = low.loc[low.betel == 'No', 'mean_p']
print(f'\nlow-risk patients: chewers n={len(a)} mean_p={a.mean():.3f} | '
      f'non-chewers n={len(b)} mean_p={b.mean():.3f}')
if len(a) > 3:
    print('Mann-Whitney (patient level) p =',
          f'{mannwhitneyu(a, b, alternative="greater").pvalue:.3g}')

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

oof = pd.read_csv(f'{TAB}/rq1_oof_predictions.csv')
rows = []
for site, g in oof.groupby('site'):
    y, yh = g.y.values, (g.p.values >= 0.5).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y, yh, average='binary', zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    rows.append(dict(site=site, precision=pr, recall=rc, f1=f1,
                     tn=tn, fp=fp, fn=fn, tp=tp))
out = pd.DataFrame(rows)
out.to_csv(f'{TAB}/rq1_precision_recall_confusion.csv', index=False)
print(out.round(3).to_string(index=False))

In [ ]:
rq3 = pd.read_csv(f'{F.TAB}/rq3_fewshot_patientboot.csv')
print(rq3[['target','k','k_actual','auc','recall','precision','f1','fn','tp']]
      .round(3).to_string(index=False))
print()
print(rq3.groupby(['target','k'])[['auc','recall']].mean().round(3))

In [ ]:
import pandas as pd, glob

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

hits = (glob.glob('/content/drive/MyDrive/**/rq3_fewshot_patientboot.csv', recursive=True)
        + glob.glob('/content/**/rq3_fewshot_patientboot.csv', recursive=True))
print('found:', hits)

if hits:
    rq3 = pd.read_csv(hits[0])
    print('\ncolumns:', list(rq3.columns), '\n')
    cols = [c for c in ['target','k','k_actual','seed','auc','recall',
                        'precision','f1','fn','tp'] if c in rq3.columns]
    print(rq3[cols].round(3).to_string(index=False))
    print()
    agg = [c for c in ['auc','recall'] if c in rq3.columns]
    print(rq3.groupby(['target','k'])[agg].mean().round(3))
else:
    print('not found — check the tables folder in your Drive for the file name')

In [ ]:
# ── Figures 4 and 5: ROC curves and confusion matrices ─────────────────────
# Uses predictions your RQ4 run already saved. No retraining.

import glob, os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception:
    pass

files = glob.glob('/content/drive/MyDrive/**/predictions_*_to_*.csv', recursive=True)
files = sorted(set(files))
print('prediction files found:')
for f in files:
    print('   ', f)
assert files, 'no prediction files found — re-run R.run_rq4() first'

OUT = os.path.dirname(files[0])
by_site = {os.path.basename(f).split('_to_')[-1][:-4]: f for f in files}

PAIRS = [('LKA_N',      'Within country  (LKA_R \u2192 LKA_N)',  '#2A6FB0'),
         ('MYS_sample', 'Across countries  (LKA_R \u2192 MYS)',  '#C2561E')]
PAIRS = [p for p in PAIRS if p[0] in by_site]

# ── Figure 4 ───────────────────────────────────────────────────────────────
plt.figure(figsize=(6.2, 5.4))
for key, label, col in PAIRS:
    d = pd.read_csv(by_site[key])
    fpr, tpr, _ = roc_curve(d.y, d.p)
    plt.plot(fpr, tpr, color=col, lw=2.2,
             label='%s   AUC = %.3f' % (label, roc_auc_score(d.y, d.p)))
plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=.5, label='Chance')
plt.xlim(0, 1); plt.ylim(0, 1.02)
plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
plt.title('ROC curves for cross-site transfer')
plt.legend(loc='lower right', frameon=False, fontsize=9)
plt.tight_layout(); plt.savefig(OUT + '/fig4_roc.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Figure 5 ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(PAIRS), figsize=(4.6 * len(PAIRS), 4.3))
axes = np.atleast_1d(axes)
for ax, (key, label, col) in zip(axes, PAIRS):
    d = pd.read_csv(by_site[key])
    cm = confusion_matrix(d.y, (d.p >= 0.5).astype(int), labels=[0, 1])
    ax.imshow(cm, cmap='Blues', vmin=0)
    for i in (0, 1):
        for j in (0, 1):
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=18,
                    color='white' if cm[i, j] > cm.max() * .6 else '#16202E')
    tn, fp, fn, tp = cm.ravel()
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Predicted\nlow-risk', 'Predicted\nhigh-risk'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['True\nlow-risk', 'True\nhigh-risk'])
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('recall %.3f    precision %.3f' %
                  (tp / max(tp + fn, 1), tp / max(tp + fp, 1)), fontsize=9)
plt.tight_layout(); plt.savefig(OUT + '/fig5_confusion.png', dpi=200, bbox_inches='tight')
plt.show()

# ── verification ───────────────────────────────────────────────────────────
print('\n--- check these against the report ---')
for key, label, _ in PAIRS:
    d = pd.read_csv(by_site[key])
    tn, fp, fn, tp = confusion_matrix(d.y, (d.p >= 0.5).astype(int), labels=[0, 1]).ravel()
    print('%-34s n=%4d  AUC %.3f   tn %3d  fp %3d  fn %3d  tp %3d   recall %.3f'
          % (label, len(d), roc_auc_score(d.y, d.p), tn, fp, fn, tp, tp / max(tp + fn, 1)))
print('\nsaved to', OUT)

In [ ]:
import pandas as pd
rows = []

In [ ]:
!ls -la /content

from google.colab import drive
drive.mount('/content/drive')

!find /content/drive/MyDrive -maxdepth 4 \
   \( -iname "*.pt" -o -iname "*.pth" -o -iname "*.pkl" \
      -o -iname "*pred*" -o -iname "*.npy" -o -iname "*result*" \) 2>/dev/null | head -40

In [ ]:
import pandas as pd, glob, os

D = "/content/drive/MyDrive/oral_cancer_crosssite/outputs"
print(os.listdir(D))
print(sorted(os.listdir(f"{D}/tables")))

for f in sorted(glob.glob(f"{D}/tables/*.csv")):
    df = pd.read_csv(f)
    print("\n===", os.path.basename(f), df.shape)
    print(list(df.columns))
    print(df.head(3).to_string())

In [ ]:
import pandas as pd
pd.set_option("display.width", 250, "display.max_columns", 50)
T = "/content/drive/MyDrive/oral_cancer_crosssite/outputs/tables"

print("===== RQ2: all twelve transfers =====")
print(pd.read_csv(f"{T}/rq2_matrix_patientboot.csv")[
    ["train","test","cross","auc","lo","hi","precision","recall","f1",
     "macro_f1","accuracy","tn","fp","fn","tp","n","n_patients"]].round(3).to_string(index=False))

print("\n===== RQ1: within-site, per site =====")
print(pd.read_csv(f"{T}/rq1_precision_recall_confusion.csv").round(3).to_string(index=False))
print(pd.read_csv(f"{T}/rq1_within_site_patientboot.csv")[
    ["site","auc","accuracy","precision","recall","f1","macro_f1","n","n_patients"]].round(3).to_string(index=False))

print("\n===== RQ1: baselines at LKA_R =====")
print(pd.read_csv(f"{T}/rq1_within_site_R.csv").round(3).to_string(index=False))

print("\n===== RQ4: referral =====")
print(pd.read_csv(f"{T}/rq4_risk_coverage_patientboot.csv")[
    ["comparison","train","test","referral_rate","accuracy","high_risk_recall",
     "n_retained","missed_high_risk"]].round(3).to_string(index=False))

In [ ]:
import pandas as pd
T = "/content/drive/MyDrive/oral_cancer_crosssite/outputs/tables"
df = pd.read_csv(f"{T}/predictions_LKA_R_to_MYS_sample.csv").rename(columns={"y":"y_true","p":"y_score"})
df["experiment"] = "LKA_R->MYS_sample"
df.to_csv("predictions.csv", index=False)
!python /content/metrics_precision_recall.py predictions.csv

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, confusion_matrix

T = "/content/drive/MyDrive/oral_cancer_crosssite/outputs/tables"

def metrics(y, p, thr=0.5):
    tn, fp, fn, tp = confusion_matrix(y, (p >= thr).astype(int), labels=[0,1]).ravel()
    prec = tp/(tp+fp) if tp+fp else np.nan
    rec  = tp/(tp+fn) if tp+fn else np.nan
    return dict(auc=roc_auc_score(y, p), precision=prec, recall=rec,
                f1=2*prec*rec/(prec+rec) if prec+rec else np.nan,
                tp=tp, fp=fp, fn=fn, tn=tn)

def boot_ci(df, key, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    groups = {k: g for k, g in df.groupby("patient_id")}
    ids = list(groups); vals = []
    for _ in range(n):
        b = pd.concat([groups[i] for i in rng.choice(ids, len(ids))])
        if b.y.nunique() < 2: continue
        vals.append(metrics(b.y.values, b.p.values)[key])
    return np.nanpercentile(vals, [2.5, 97.5])

runs = {
    "LKA_R -> LKA_N":      pd.read_csv(f"{T}/predictions_LKA_R_to_LKA_N.csv"),
    "LKA_R -> MYS_sample": pd.read_csv(f"{T}/predictions_LKA_R_to_MYS_sample.csv"),
}
oof = pd.read_csv(f"{T}/rq1_oof_predictions.csv")
for site, g in oof.groupby("site"):
    runs[f"{site} within-site"] = g

rows = []
for name, df in runs.items():
    m = metrics(df.y.values, df.p.values)
    plo, phi = boot_ci(df, "precision")
    rlo, rhi = boot_ci(df, "recall")
    rows.append([name, len(df), df.patient_id.nunique(),
                 round(m["auc"],3), round(m["precision"],3), f"{plo:.3f}–{phi:.3f}",
                 round(m["recall"],3), f"{rlo:.3f}–{rhi:.3f}", round(m["f1"],3),
                 f'{m["tp"]}/{m["fp"]}/{m["fn"]}/{m["tn"]}'])

out = pd.DataFrame(rows, columns=["experiment","n","patients","AUC","precision","prec 95% CI",
                                  "recall","recall 95% CI","F1","TP/FP/FN/TN"])
print(out.to_string(index=False))
out.to_csv(f"{T}/precision_recall_with_ci.csv", index=False)

In [ ]:
print(open("/content/drive/MyDrive/oral_cancer_crosssite/src/pipeline.py").read())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob, json, os
ROOT = "/content/drive/MyDrive"
keys = ["make_cnn_predictor", "RandomForest", "optim.", "ColorJitter", "Resize", "lr="]

files = glob.glob(f"{ROOT}/**/*.py", recursive=True) + glob.glob(f"{ROOT}/**/*.ipynb", recursive=True)
files = [f for f in files if "oral" in f.lower() or "pipeline" in f.lower()]
print("candidate files:", *files, sep="\n  ")

for f in files:
    if f.endswith(".py"):
        src = open(f).read()
    else:
        nb = json.load(open(f))
        src = "\n# ---- cell ----\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    if any(k in src for k in keys):
        print("\n" + "=" * 80 + f"\nFOUND IN: {f}\n" + "=" * 80)
        lines = src.splitlines()
        for i, line in enumerate(lines):
            if any(k in line for k in keys + ["def make_", "transforms.", "criterion", "Loss(", "n_estimators", "max_depth", "class_weight"]):
                print("\n".join(lines[max(0, i-3): i+12]))
                print("    ...")

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score
T = "/content/drive/MyDrive/oral_cancer_crosssite/outputs/tables"

# ---------- Table 4: lesion type vs all low-risk, same LKA_R-trained model ----------
df = pd.read_csv(f"{T}/predictions_trainR.csv")
print(df.groupby(["site","risk_label"]).clinical_dx.value_counts().to_string(), "\n")

def boot(sub, n=2000, seed=0):
    rng=np.random.default_rng(seed); g={k:v for k,v in sub.groupby("patient_id")}; ids=list(g); a=[]
    for _ in range(n):
        b=pd.concat([g[i] for i in rng.choice(ids,len(ids))])
        if b.risk_label.nunique()==2: a.append(roc_auc_score(b.risk_label,b.prob))
    return np.percentile(a,[2.5,97.5])

for site, key in [("LKA_N","fibrosis"),("LKA_N","leukoplakia"),("LKA_N","lichen"),("MYS_sample","lichen"),
                  ("SAMPLE_Malaysia","lichen")]:
    s = df[df.site==site]
    if s.empty: continue
    sub = pd.concat([s[(s.risk_label==1) & s.clinical_dx.str.lower().str.contains(key)], s[s.risk_label==0]])
    lo,hi = boot(sub)
    print(f"{site:16s} {key:12s} n={len(sub):4d}  AUC={roc_auc_score(sub.risk_label,sub.prob):.3f}  ({lo:.3f}–{hi:.3f})")

# ---------- Table 5: adaptation ----------
print("\n", pd.read_csv(f"{T}/rq3_fewshot_curve_patientboot.csv").to_string(index=False))
r = pd.read_csv(f"{T}/rq3_fewshot_patientboot.csv")
print("\nrecall by target and k (mean over seeds):")
print(r.groupby(["target","k"]).recall.mean().round(3).unstack().to_string())

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score
T = "/content/drive/MyDrive/oral_cancer_crosssite/outputs/tables"
df = pd.read_csv(f"{T}/predictions_trainR.csv")

def boot(sub, n=2000, seed=0):
    rng=np.random.default_rng(seed); g={k:v for k,v in sub.groupby("patient_id")}; ids=list(g); a=[]
    for _ in range(n):
        b=pd.concat([g[i] for i in rng.choice(ids,len(ids))])
        if b.risk_label.nunique()==2: a.append(roc_auc_score(b.risk_label,b.prob))
    return np.percentile(a,[2.5,97.5]) if a else (np.nan,np.nan)

print("=== Table 4 ===")
for site, key, label in [("LKA_N","osf","Submucous fibrosis"),("LKA_N","leukoplakia","Leukoplakia"),
                         ("LKA_N","olp","Lichen planus"),("MYS_sample","lichen","Lichen planus (MYS)")]:
    s = df[df.site==site]
    pos = s[(s.risk_label==1) & s.clinical_dx.str.lower().str.contains(key)]
    sub = pd.concat([pos, s[s.risk_label==0]])
    lo,hi = boot(sub)
    print(f"{label:22s} n={len(sub):4d}  AUC={roc_auc_score(sub.risk_label,sub.prob):.3f}  ({lo:.3f}–{hi:.3f})")

print("\n=== Table 5 ===")
print(pd.read_csv(f"{T}/rq3_fewshot_curve_patientboot.csv").to_string(index=False))
r = pd.read_csv(f"{T}/rq3_fewshot_patientboot.csv")
print("\nrecall by target and k (mean over seeds):")
print(r.groupby(["target","k"]).recall.mean().round(3).unstack().to_string())

In [ ]:
r = pd.read_csv("/content/drive/MyDrive/oral_cancer_crosssite/outputs/tables/rq3_fewshot_patientboot.csv")
print(r[(r.target=="LKA_S") & (r.k==10)].auc.mean())

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

r = pd.read_csv("/content/drive/MyDrive/oral_cancer_crosssite/outputs/tables/rq3_fewshot_patientboot.csv")
k10 = r[(r.target=="LKA_S") & (r.k==10)]
print(k10[["seed","auc"]].to_string(index=False))
print("mean:", round(k10.auc.mean(), 5), "| target: 0.63184 | meets target:", k10.auc.mean() >= 0.631838)